In [57]:
!pip install -q openai

In [58]:
import requests
from google.colab import userdata

API_KEY = userdata.get("NEXUS_API_KEY")
BASE_URL = "https://nexusapi.navigatelabs.ai"

response = requests.get(
    f"{BASE_URL}/v1/models",
    headers={
        "Authorization": f"Bearer {API_KEY}"
    },
    timeout=30
)

print("Status Code:", response.status_code)
print("Response:")
print(response.text[:5000])

Status Code: 200
Response:
{"data":[{"id":"nano-banana","object":"model","created":1677610602,"owned_by":"openai","mode":"image_generation","max_input_tokens":65536,"max_output_tokens":32768},{"id":"whisper-1","object":"model","created":1677610602,"owned_by":"openai","mode":"audio_transcription"},{"id":"nova-micro","object":"model","created":1677610602,"owned_by":"openai","mode":"chat","max_input_tokens":128000,"max_output_tokens":10000},{"id":"text-embedding-3-small","object":"model","created":1677610602,"owned_by":"openai","mode":"embedding","max_input_tokens":8191},{"id":"gpt-4o-mini-tts","object":"model","created":1677610602,"owned_by":"openai","mode":"audio_speech"},{"id":"gpt-4.1-nano","object":"model","created":1677610602,"owned_by":"openai","mode":"chat","max_input_tokens":1047576,"max_output_tokens":32768}],"object":"list"}


In [59]:
from getpass import getpass

API_KEY = getpass("Enter your Navigate Labs API key: ")

BASE_URL = "https://nexusapi.navigatelabs.ai"

print("API key loaded:", bool(API_KEY))

Enter your Navigate Labs API key: ··········
API key loaded: True


In [60]:
import requests
from google.colab import userdata

API_KEY = userdata.get("NEXUS_API_KEY")
BASE_URL = "https://nexusapi.navigatelabs.ai"

response = requests.post(
    f"{BASE_URL}/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "nova-micro",
        "messages": [
            {
                "role": "user",
                "content": "Explain what an AI document management assistant does in two sentences."
            }
        ],
        "max_tokens": 150
    },
    timeout=60
)

print("Status Code:", response.status_code)
print(response.text[:3000])

Status Code: 200
{"id":"chatcmpl-50c00b9f-da57-485c-943f-6d9372dd74d3","created":1795104749,"model":"nova-micro","object":"chat.completion","choices":[{"finish_reason":"stop","index":0,"message":{"content":"An AI document management assistant automates the organization, retrieval, and processing of documents, significantly enhancing efficiency and accuracy in document handling through advanced algorithms and machine learning techniques. It also supports users by providing intelligent search capabilities and streamlining workflows through intelligent routing and classification.","role":"assistant"}}],"usage":{"completion_tokens":52,"prompt_tokens":12,"total_tokens":64,"completion_tokens_details":{"reasoning_tokens":0,"text_tokens":52},"prompt_tokens_details":{"cached_tokens":0,"text_tokens":12,"cache_write_tokens":0,"cache_creation_tokens":0},"cache_creation_input_tokens":0,"cache_read_input_tokens":0}}


In [61]:
import requests

response = requests.post(
    f"{BASE_URL}/v1/embeddings",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "text-embedding-3-small",
        "input": "Aadhaar proof of address documents"
    },
    timeout=60
)

print("Status Code:", response.status_code)

if response.status_code == 200:
    result = response.json()
    embedding = result["data"][0]["embedding"]

    print("Embedding API working!")
    print("Embedding dimensions:", len(embedding))
else:
    print(response.text[:2000])

Status Code: 200
Embedding API working!
Embedding dimensions: 1536


In [62]:
import requests
import json
import time
import re

from getpass import getpass

# Navigate Labs configuration
BASE_URL = "https://nexusapi.navigatelabs.ai"

# Reuse the company key if already loaded
if "API_KEY" not in globals() or not API_KEY:
    API_KEY = getpass("Enter Navigate Labs API key: ")

NEXUS_API_KEY = API_KEY

CHAT_MODEL = "nova-micro"
EMBED_MODEL = "text-embedding-3-small"


# TEXT GENERATION
def nexus_generate(prompt, json_mode=False):

    if json_mode:
        prompt += "\n\nReturn ONLY valid JSON. Do not use Markdown."

    response = requests.post(
        f"{BASE_URL}/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {NEXUS_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": CHAT_MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0.2,
            "max_tokens": 2048
        },
        timeout=60
    )

    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]


# TEXT EMBEDDINGS
def nexus_embedding(text):

    response = requests.post(
        f"{BASE_URL}/v1/embeddings",
        headers={
            "Authorization": f"Bearer {NEXUS_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": EMBED_MODEL,
            "input": text
        },
        timeout=60
    )

    response.raise_for_status()

    return response.json()["data"][0]["embedding"]


print("Navigate Labs API functions initialized!")

Navigate Labs API functions initialized!


In [63]:
# Chat test
result = nexus_generate(
    "Reply with exactly: Navigate Labs is working."
)

print("CHAT TEST:", result)


# Embedding test
test_embedding = nexus_embedding(
    "Government document management assistant"
)

print("\nEMBEDDING TEST")
print("Dimensions:", len(test_embedding))

CHAT TEST: Navigate Labs is working.

EMBEDDING TEST
Dimensions: 1536


In [64]:
# Install chromadb and initialize ChromaDB client
!pip install -q chromadb

import os
import chromadb

# Pointing to the persistent database path on Google Drive
NEW_CHROMA_PATH = "/content/drive/MyDrive/AI_Document_Assistant/chroma_db"
os.makedirs(NEW_CHROMA_PATH, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=NEW_CHROMA_PATH)

nexus_collection = chroma_client.get_or_create_collection(
    name="government_knowledge_nexus",
    embedding_function=None,
    metadata={"hnsw:space": "cosine"}
)

print("Navigate Labs collection ready!")
try:
    # Use the clean collection created earlier in the notebook if it exists
    collection = chroma_client.get_collection(name="government_knowledge_clean")
    print("Existing clean collection:", collection.count())
except Exception:
    print("Existing clean collection: Not found/Not yet initialized")
print("New Nexus collection:", nexus_collection.count())

Navigate Labs collection ready!
Existing clean collection: Not found/Not yet initialized
New Nexus collection: 0


In [180]:
print("========== KNOWLEDGE BASE STATUS ==========")

print("Collection name:", nexus_collection.name)
print("Total indexed chunks:", nexus_collection.count())

print("\n========== STORED GOVERNMENT DOCUMENTS ==========")

import os

gov_docs_folder = "/content/drive/MyDrive/AI_Document_Assistant/government_documents"

print("Folder exists:", os.path.exists(gov_docs_folder))

if os.path.exists(gov_docs_folder):
    print("Files:", os.listdir(gov_docs_folder))

========== KNOWLEDGE BASE STATUS ==========
Collection name: government_knowledge_nexus
Total indexed chunks: 0

========== STORED GOVERNMENT DOCUMENTS ==========
Folder exists: True
Files: []


In [181]:
import os

print("========== CHROMADB LOCATION CHECK ==========")

possible_path = (
    "/content/drive/MyDrive/"
    "AI_Document_Assistant/chroma_db"
)

print("Expected folder:", possible_path)
print("Folder exists:", os.path.exists(possible_path))

if os.path.exists(possible_path):
    for root, dirs, files in os.walk(possible_path):
        for file in files:
            path = os.path.join(root, file)
            print(path, "| Size:", os.path.getsize(path), "bytes")

print("\n========== CURRENT COLLECTION ==========")

print("Collection:", nexus_collection.name)
print("Count:", nexus_collection.count())

print("\n========== AVAILABLE CLIENT VARIABLES ==========")

for name, value in globals().copy().items():
    if "client" in name.lower() or "chroma" in name.lower():
        print(name, ":", type(value).__name__)

========== CHROMADB LOCATION CHECK ==========
Expected folder: /content/drive/MyDrive/AI_Document_Assistant/chroma_db
Folder exists: True
/content/drive/MyDrive/AI_Document_Assistant/chroma_db/chroma.sqlite3 | Size: 188416 bytes

========== CURRENT COLLECTION ==========
Collection: government_knowledge_nexus
Count: 0

========== AVAILABLE CLIENT VARIABLES ==========
chromadb : module
NEW_CHROMA_PATH : str
chroma_client : Client
local_chroma_path : str
original_client : Client
local_client : Client
drive_client : Client
client_name : str
client : Client
local_chroma_client : Client
drive_chroma_path : str
drive_chroma_client : Client
temp_client : Client


In [182]:
print("========== CHROMADB CLIENT COMPARISON ==========")

client_names = [
    "chroma_client",
    "original_client",
    "local_client",
    "drive_client",
    "client",
    "local_chroma_client",
    "drive_chroma_client",
    "temp_client"
]

for name in client_names:
    obj = globals().get(name)

    if obj is None:
        continue

    print(f"\nClient: {name}")
    print("Type:", type(obj).__name__)

    try:
        collections = obj.list_collections()

        if not collections:
            print("No collections found.")

        for collection in collections:
            if isinstance(collection, str):
                collection_name = collection
            else:
                collection_name = collection.name

            print("Collection:", collection_name)

            try:
                col = obj.get_collection(collection_name)
                print("Document chunks:", col.count())
            except Exception as e:
                print("Could not inspect:", str(e))

    except Exception as e:
        print("Client inspection error:", str(e))

print("\n========== CHROMADB PATH VARIABLES ==========")

for name in [
    "NEW_CHROMA_PATH",
    "local_chroma_path",
    "drive_chroma_path"
]:
    if name in globals():
        print(name, "=", globals()[name])

========== CHROMADB CLIENT COMPARISON ==========

Client: chroma_client
Type: Client
Collection: government_knowledge
Document chunks: 0

Client: original_client
Type: Client
Collection: government_knowledge
Document chunks: 0

Client: local_client
Type: Client
Collection: government_knowledge
Document chunks: 0

Client: drive_client
Type: Client
Collection: government_knowledge_nexus
Document chunks: 0

Client: client
Type: Client
Collection: government_knowledge_nexus
Document chunks: 0

Client: local_chroma_client
Type: Client
Collection: government_knowledge
Document chunks: 0

Client: drive_chroma_client
Type: Client
Collection: government_knowledge_nexus
Document chunks: 0

Client: temp_client
Type: Client
Collection: government_knowledge_nexus
Document chunks: 0

========== CHROMADB PATH VARIABLES ==========
NEW_CHROMA_PATH = /content/drive/MyDrive/AI_Document_Assistant/chroma_db
local_chroma_path = /content/chroma_db
drive_chroma_path = /content/drive/MyDrive/AI_Document_Assist

In [183]:
import sqlite3
import os

db_path = (
    "/content/drive/MyDrive/AI_Document_Assistant/"
    "chroma_db/chroma.sqlite3"
)

print("========== DATABASE FILE ==========")
print("Exists:", os.path.exists(db_path))
print("Size:", os.path.getsize(db_path), "bytes")

connection = sqlite3.connect(
    f"file:{db_path}?mode=ro",
    uri=True
)

cursor = connection.cursor()

print("\n========== DATABASE TABLES ==========")

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
""")

tables = [row[0] for row in cursor.fetchall()]

for table in tables:
    print("\nTable:", table)

    try:
        cursor.execute(f'SELECT COUNT(*) FROM "{table}"')
        print("Rows:", cursor.fetchone()[0])
    except Exception as e:
        print("Inspection error:", e)

print("\n========== COLLECTION RECORDS ==========")

if "collections" in tables:
    cursor.execute("SELECT * FROM collections")
    for row in cursor.fetchall():
        print(row)

connection.close()
print("\nRead-only inspection completed.")

========== DATABASE FILE ==========
Exists: True
Size: 188416 bytes

========== DATABASE TABLES ==========

Table: migrations
Rows: 18

Table: acquire_write
Rows: 1

Table: collection_metadata
Rows: 1

Table: segment_metadata
Rows: 0

Table: tenants
Rows: 1

Table: databases
Rows: 1

Table: collections
Rows: 1

Table: maintenance_log
Rows: 0

Table: segments
Rows: 2

Table: embeddings
Rows: 0

Table: embedding_metadata
Rows: 0

Table: max_seq_id
Rows: 0

Table: embedding_fulltext_search
Rows: 0

Table: embedding_fulltext_search_data
Rows: 2

Table: embedding_fulltext_search_idx
Rows: 0

Table: embedding_fulltext_search_content
Rows: 0

Table: embedding_fulltext_search_docsize
Rows: 0

Table: embedding_fulltext_search_config
Rows: 1

Table: embedding_metadata_array
Rows: 0

Table: embeddings_queue
Rows: 0

Table: embeddings_queue_config
Rows: 1

========== COLLECTION RECORDS ==========
('f03f4fd6-de65-42d4-b71b-bf9e7b51d83e', 'government_knowledge_nexus', None, '00000000-0000-0000-0000-

In [184]:
import os

search_roots = [
    "/content/drive/MyDrive",
    "/content"
]

found_databases = []
found_pdfs = []

for root_path in search_roots:

    for root, dirs, files in os.walk(root_path):

        # Avoid unnecessary system folders
        dirs[:] = [
            d for d in dirs
            if d not in ["node_modules", ".git"]
        ]

        for filename in files:

            full_path = os.path.join(root, filename)

            if filename == "chroma.sqlite3":
                found_databases.append(full_path)

            elif filename.lower().endswith(".pdf"):
                found_pdfs.append(full_path)

print("========== CHROMADB FILES ==========")

for path in sorted(set(found_databases)):
    print(path, "|", os.path.getsize(path), "bytes")

print("\n========== GOVERNMENT/PERSONAL PDF FILES ==========")

for path in sorted(set(found_pdfs)):
    print(path)

print("\nTotal ChromaDB files:", len(set(found_databases)))
print("Total PDF files:", len(set(found_pdfs)))

========== CHROMADB FILES ==========
/content/chroma_db/chroma.sqlite3 | 188416 bytes
/content/drive/MyDrive/AI_Document_Assistant/chroma_db/chroma.sqlite3 | 188416 bytes

========== GOVERNMENT/PERSONAL PDF FILES ==========
/content/drive/MyDrive/AI_Document_Assistant/personal_documents/sample_certificate.pdf
/content/sample_certificate.pdf

Total ChromaDB files: 2
Total PDF files: 2


In [186]:
import inspect

# We check only the functions that are actually defined in our session namespace
functions_to_check = []
for name in [
    "generate_dashboard",
    "upload_personal",
    "search_documents",
    "save_document_edits",
    "agent_assistant"
]:
    if name in globals():
        functions_to_check.append(globals()[name])

print("========== CALLBACK STORAGE REFERENCES ==========")

for fn in functions_to_check:
    print(f"\nFunction: {fn.__name__}")
    print("Referenced names:", fn.__code__.co_names)

    namespace = fn.__globals__
    for name in fn.__code__.co_names:
        if name in namespace:
            value = namespace[name]
            if any(word in name.lower() for word in [
                "db", "database", "folder", "document",
                "path", "load", "save"
            ]):
                print(
                    name, "=",
                    value if isinstance(value, str)
                    else type(value).__name__
                )

========== CALLBACK STORAGE REFERENCES ==========

Function: generate_dashboard
Referenced names: ('check_expiry', 'len', 'sum', 'reversed')

Function: upload_personal
Referenced names: ('format_expiry_table', 'check_expiry', 'generate_dashboard', 'isinstance', 'list', 'save_personal_file', 'process_personal_document', 'append', 'os', 'path', 'basename', 'Exception', 'print', 'str', 'len', 'join')
process_personal_document = function
path = /content/sample_certificate.pdf

Function: search_documents
Referenced names: ('check_expiry', 'strip', 'lower', 'append', 'format_expiry_table')

Function: save_document_edits
Referenced names: ('generate_dashboard', 'hasattr', 'values', 'tolist', 'enumerate', 'len', 'str', 'strip', 'check_expiry', 'format_expiry_table', 'Exception')

Function: agent_assistant
Referenced names: ('lower', 'any', 'personal_document_tool', 'renewal_guidance_tool', 'expiry_tracking_tool', 'government_rag_tool', 'gemini_generate')
personal_document_tool = function


In [179]:
import os
import time
import uuid

gov_docs_folder = "/content/drive/MyDrive/AI_Document_Assistant/government_documents"

if not os.path.exists(gov_docs_folder):
    print(f"Warning: The folder {gov_docs_folder} does not exist. Creating it now...")
    os.makedirs(gov_docs_folder, exist_ok=True)

pdf_files = [os.path.join(gov_docs_folder, f) for f in os.listdir(gov_docs_folder) if f.lower().endswith('.pdf')]

if not pdf_files:
    print("No PDFs found to parse in the government documents folder. Please upload your PDFs to that folder or via the Gradio interface.")
else:
    print(f"Found {len(pdf_files)} PDF(s) to process.")
    try:
        # Process files using your registered helper function
        result = add_to_knowledge_base(pdf_files)
        print(result)
        print(f"Current chunks in Navigate Labs Nexus Collection: {nexus_collection.count()}")
    except Exception as e:
        print(f"An error occurred during indexing: {str(e)}")

No PDFs found to parse in the government documents folder. Please upload your PDFs to that folder or via the Gradio interface.


In [66]:
import os
import glob

print("Searching for database files across paths...")

# Look for any .sqlite or db directories
search_paths = [
    "/content/**/*.db",
    "/content/**/*.sqlite*",
    "/content/drive/MyDrive/AI_Document_Assistant/**/*.db",
    "/content/drive/MyDrive/AI_Document_Assistant/**/*.sqlite*"
]

found_files = []
for path in search_paths:
    found_files.extend(glob.glob(path, recursive=True))

print(f"Found {len(found_files)} potential DB files:")
for f in found_files:
    print("-", f)

# Also let's inspect the subdirectories in your Project Storage to see if there is another backup folder
project_storage_path = "/content/drive/MyDrive/AI_Document_Assistant"
if os.path.exists(project_storage_path):
    print(f"\nContents of {project_storage_path}:")
    for item in os.listdir(project_storage_path):
        full_path = os.path.join(project_storage_path, item)
        is_dir = "[DIR]" if os.path.isdir(full_path) else "[FILE]"
        print(f" {is_dir} {item}")

Searching for database files across paths...
Found 3 potential DB files:
- /content/chroma_db/chroma.sqlite3
- /content/drive/MyDrive/AI_Document_Assistant/chroma_db/chroma.sqlite3
- /content/drive/MyDrive/AI_Document_Assistant/chroma_db/chroma.sqlite3

Contents of /content/drive/MyDrive/AI_Document_Assistant:
 [DIR] government_documents
 [DIR] chroma_db
 [DIR] personal_documents


In [67]:
# Test Navigate Labs retrieval

test_question = "What documents are accepted as Proof of Identity for Aadhaar enrolment?"

query_embedding = nexus_embedding(test_question)

results = nexus_collection.query(
    query_embeddings=[query_embedding],
    n_results=4,
    include=["documents", "metadatas", "distances"]
)

print("Retrieved chunks:", len(results["documents"][0]))

for i, doc in enumerate(results["documents"][0]):

    metadata = results["metadatas"][0][i]

    print("\n" + "=" * 60)
    print("Source:", metadata.get("source", "Unknown"))
    print("Page:", metadata.get("page", "Unknown"))
    print("Distance:", results["distances"][0][i])
    print("Content:", doc[:700])

Retrieved chunks: 0


In [68]:
def retrieve_information(question, n_results=4):

    query_embedding = nexus_embedding(question)

    results = nexus_collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    for i, doc in enumerate(results["documents"][0]):

        metadata = results["metadatas"][0][i]

        retrieved_chunks.append({
            "content": doc,
            "source": metadata.get("source", "Unknown"),
            "page": metadata.get("page", "Unknown"),
            "distance": results["distances"][0][i]
        })

    return retrieved_chunks


print("Navigate Labs retrieval function initialized!")

Navigate Labs retrieval function initialized!


In [69]:
def rag_answer(question):

    retrieved_chunks = retrieve_information(question)

    if not retrieved_chunks:
        return "I couldn't find relevant information in the government documents."

    context = ""

    for chunk in retrieved_chunks:
        context += (
            f"\nSource: {chunk['source']}"
            f"\nPage: {chunk['page']}"
            f"\nContent: {chunk['content']}\n"
        )

    prompt = f"""
You are an AI Government Service Assistant.

Answer the user's question using ONLY the official document context below.

Instructions:
1. Give a clear and simple answer.
2. Use bullet points where helpful.
3. Mention the source PDF and page number.
4. Do not invent information.
5. If the answer is unavailable in the context, clearly say so.

OFFICIAL DOCUMENT CONTEXT:
{context}

USER QUESTION:
{question}
"""

    return nexus_generate(prompt)


print("Navigate Labs RAG answer function initialized!")

Navigate Labs RAG answer function initialized!


In [70]:
answer = rag_answer(
    "What documents are accepted as Proof of Identity for Aadhaar enrolment for adults?"
)

print(answer)

I couldn't find relevant information in the government documents.


In [71]:
def extract_document_details(text):

    prompt = f"""
Extract information from the following document.

Return ONLY valid JSON with these keys:
{{
    "document_type": "",
    "issue_date": "",
    "expiry_date": "",
    "confidence": ""
}}

Rules:
- Use YYYY-MM-DD for dates.
- If a date is unavailable, return an empty string.
- Do not guess missing information.
- Identify the document type accurately.

DOCUMENT TEXT:
{text}
"""

    response = nexus_generate(prompt, json_mode=True)

    # Extract JSON safely
    try:
        match = re.search(r'\{.*\}', response, re.DOTALL)

        if not match:
            raise ValueError("No JSON object found in AI response.")

        details = json.loads(match.group())

        return details

    except (json.JSONDecodeError, ValueError) as e:
        print("Extraction error:", e)
        print("AI response:", response)
        return None


print("Navigate Labs document extraction function initialized!")

Navigate Labs document extraction function initialized!


In [72]:
test_text = """
Vendor Compliance Certificate

Issue Date: 18 November 2026
Expiry Date: 17 November 2027

This certificate confirms that the vendor
has successfully completed compliance verification.
"""

test_details = extract_document_details(test_text)

print("Extracted Document Details:")
print(json.dumps(test_details, indent=4))

Extracted Document Details:
{
    "document_type": "Vendor Compliance Certificate",
    "issue_date": "2026-11-18",
    "expiry_date": "2027-11-17",
    "confidence": ""
}


In [89]:
import os
import shutil
import sqlite3
from datetime import datetime, date

def process_personal_document(file_path, documents):

    # 1. Validate uploaded file
    if not file_path or not os.path.exists(file_path):
        raise FileNotFoundError("Uploaded file not found.")

    # 2. Extract document text
    extracted_text = extract_document_text(file_path)

    if not extracted_text or not extracted_text.strip():
        raise ValueError(
            "Could not extract text from the uploaded document."
        )

    # 3. Extract metadata using Navigate Labs
    details = extract_document_details(extracted_text)

    if not details:
        raise ValueError("AI document extraction failed.")

    # 4. Calculate expiry status
    expiry_date = details.get("expiry_date", "")

    if expiry_date:
        try:
            expiry = datetime.strptime(
                expiry_date, "%Y-%m-%d"
            ).date()

            today = date.today()

            if expiry < today:
                status = "Expired"
            elif (expiry - today).days <= 30:
                status = "Expiring Soon"
            else:
                status = "Valid"

        except ValueError:
            status = "Unknown"
    else:
        status = "No Expiry Date"

    # 5. Copy uploaded file permanently into Google Drive
    os.makedirs(PERSONAL_FOLDER, exist_ok=True)

    original_name = os.path.basename(file_path)
    name, extension = os.path.splitext(original_name)

    permanent_path = os.path.join(
        PERSONAL_FOLDER, original_name
    )

    counter = 1

    while os.path.exists(permanent_path):
        permanent_path = os.path.join(
            PERSONAL_FOLDER,
            f"{name}_{counter}{extension}"
        )
        counter += 1

    shutil.copy2(file_path, permanent_path)

    # 6. Prepare document metadata
    document_record = {
        "document": os.path.basename(permanent_path),
        "type": details.get("document_type", "Unknown"),
        "issue_date": details.get("issue_date", ""),
        "expiry_date": expiry_date,
        "confidence": details.get("confidence", 0),
        "status": status,
        "file_path": permanent_path
    }

    # 7. Save metadata permanently in SQLite
    save_document_to_database(document_record)

    # 8. Reload records from database
    documents = load_documents_from_database()

    print("Document successfully processed!")
    print("Saved file:", permanent_path)
    print("Document type:", document_record["type"])
    print("Status:", status)

    return documents

In [90]:
print("Upload function:", callable(process_personal_document))
print("Database save function:", callable(save_document_to_database))
print("Database load function:", callable(load_documents_from_database))

print("\nDatabase records:", len(load_documents_from_database()))
print("Personal folder:", os.listdir(PERSONAL_FOLDER))

Upload function: True
Database save function: True
Database load function: True

Database records: 0
Personal folder: []


In [91]:
import os

print("Checking sample PDF files...\n")

for root, dirs, files in os.walk("/content"):
    dirs[:] = [
        d for d in dirs
        if d not in ["drive", ".cache", "chroma_db"]
    ]

    for file in files:
        if file.lower().endswith(".pdf"):
            print(os.path.join(root, file))

Checking sample PDF files...



In [92]:
!pip -q install reportlab

from reportlab.pdfgen import canvas

sample_path = "/content/sample_certificate.pdf"

pdf = canvas.Canvas(sample_path)

pdf.drawString(100, 750, "SAMPLE CERTIFICATE")
pdf.drawString(100, 710, "Document Type: Training Certificate")
pdf.drawString(100, 680, "Holder Name: Test User")
pdf.drawString(100, 650, "Issue Date: 2026-11-01")
pdf.drawString(100, 620, "Expiry Date: 2027-11-01")

pdf.save()

print("Sample PDF created:", os.path.exists(sample_path))
print("File:", sample_path)

Sample PDF created: True
File: /content/sample_certificate.pdf


In [97]:
!pip install -q pymupdf
import os
import fitz as pymupdf

# Fallback definition for extract_document_text if not already in globals
if 'extract_document_text' not in globals():
    def extract_document_text(file_path):
        try:
            doc = pymupdf.open(file_path)
            text = ""
            for page in doc:
                text += page.get_text()
            doc.close()
            return text
        except Exception as e:
            print(f"Error reading PDF: {e}")
            return ""

# Ensure documents is defined
if 'documents' not in globals():
    documents = load_documents_from_database()

documents = process_personal_document(
    "/content/sample_certificate.pdf",
    documents
)

print("\n========== UPLOAD TEST ==========")
print("Total records:", len(documents))

for doc in documents:
    print(doc)

print("\nPDF saved in Drive:",
      os.path.exists(documents[-1]["file_path"]))

print("Database records:",
      len(load_documents_from_database()))

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Document successfully processed!
Saved file: /content/drive/MyDrive/AI_Document_Assistant/personal_documents/sample_certificate.pdf
Document type: Training Certificate
Status: Valid

========== UPLOAD TEST ==========
Total records: 1
{'id': 1, 'document': 'sample_certificate.pdf', 'type': 'Training Certificate', 'issue_date': '2026-11-01', 'expiry_date': '2027-11-01', 'status': 'Valid', 'confidence': '', 'file_path': '/content/drive/MyDrive/AI_Document_Assistant/personal_documents/sample_certificate.pdf'}

PDF saved in Drive: True
Database records: 1


In [74]:
import glob
import os

# Safely define the personal documents folder path if it hasn't been defined yet
if "PERSONAL_DOCS_FOLDER" not in globals():
    PROJECT_STORAGE = "/content/drive/MyDrive/AI_Document_Assistant"
    PERSONAL_DOCS_FOLDER = os.path.join(PROJECT_STORAGE, "personal_documents")
    os.makedirs(PERSONAL_DOCS_FOLDER, exist_ok=True)

sample_files = glob.glob(
    os.path.join(PERSONAL_DOCS_FOLDER, "*.pdf")
)

print("Available sample documents:")

for i, file in enumerate(sample_files):
    print(i, os.path.basename(file))

Available sample documents:


In [75]:
import os
import json

if not sample_files:
    print("No sample files found in:", PERSONAL_DOCS_FOLDER)
    print("Please upload some personal PDF documents to this folder first, then rerun this cell!")
else:
    test_file = sample_files[0]
    print("Testing:", os.path.basename(test_file))

    extracted_text = extract_document_text(test_file)

    print("\nExtracted text preview:")
    print(extracted_text[:1000])

    print("\nNavigate Labs extraction:")
    test_details = extract_document_details(extracted_text)
    print(json.dumps(test_details, indent=4))

No sample files found in: /content/drive/MyDrive/AI_Document_Assistant/personal_documents
Please upload some personal PDF documents to this folder first, then rerun this cell!


In [76]:
def agent_assistant(question, documents):

    question_lower = question.lower()

    # 1. Personal document questions
    if any(word in question_lower for word in [
        "uploaded documents",
        "my documents",
        "personal documents",
        "show my documents",
        "list my documents"
    ]):
        tool_result = personal_document_tool(question, documents)
        selected_tool = "Personal Document Tool"

    # 2. Renewal questions
    elif any(word in question_lower for word in [
        "renew",
        "renewal",
        "reissue"
    ]):
        personal_result = personal_document_tool(
            question, documents
        )

        government_result = renewal_guidance_tool(question)

        tool_result = (
            "PERSONAL DOCUMENT INFORMATION:\n"
            + str(personal_result)
            + "\n\nGOVERNMENT RENEWAL GUIDANCE:\n"
            + str(government_result)
        )

        selected_tool = "Renewal Guidance Tool"

    # 3. Expiry questions
    elif any(word in question_lower for word in [
        "expiry",
        "expire",
        "expired",
        "validity",
        "days remaining"
    ]):
        tool_result = expiry_tracking_tool(question, documents)
        selected_tool = "Expiry Tracking Tool"

    # 4. Government information questions
    else:
        tool_result = government_rag_tool(question)
        selected_tool = "Government RAG Tool"

    prompt = f"""
You are an AI Personal Document and Government Service Assistant.

User Question:
{question}

Selected Tool:
{selected_tool}

Retrieved Information:
{tool_result}

Instructions:
- Answer clearly and simply.
- Use only retrieved information.
- Never invent expiry dates or renewal procedures.
- If renewal information is unavailable, say so.
- Preserve government PDF source references.
"""

    # Navigate Labs instead of Gemini
    answer = nexus_generate(prompt)

    return (
        f"🤖 Selected Tool: {selected_tool}\n\n"
        f"{answer}"
    )


print("Navigate Labs Agentic AI function initialized!")

Navigate Labs Agentic AI function initialized!


In [81]:
from google.colab import drive
import os

# Mount Google Drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

print("Google Drive mounted:", os.path.exists("/content/drive/MyDrive"))

# Search for the existing database
matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "personal_documents.db" in files:
        matches.append(os.path.join(root, "personal_documents.db"))

print("\nDatabase search results:")

if matches:
    for path in matches:
        print("FOUND:", path)
else:
    print("No personal_documents.db found in MyDrive.")

Google Drive mounted: True

Database search results:
No personal_documents.db found in MyDrive.


In [82]:
import os

print("========== PROJECT FOLDERS ==========")

drive_path = "/content/drive/MyDrive"

for item in os.listdir(drive_path):
    if "document" in item.lower() or "assistant" in item.lower():
        full_path = os.path.join(drive_path, item)
        print(item, "->", "Folder" if os.path.isdir(full_path) else "File")

print("\n========== DATABASE FILES ==========")

for root, dirs, files in os.walk(drive_path):
    dirs[:] = [d for d in dirs if d not in [
        ".ipynb_checkpoints", "node_modules"
    ]]

    for file in files:
        if file.endswith((".db", ".sqlite", ".sqlite3")):
            print(os.path.join(root, file))

========== PROJECT FOLDERS ==========
AI_Document_Assistant -> Folder

========== DATABASE FILES ==========
/content/drive/MyDrive/AI_Document_Assistant/chroma_db/chroma.sqlite3


In [83]:
import os

PROJECT_STORAGE = "/content/drive/MyDrive/AI_Document_Assistant"

print("========== PROJECT STORAGE ==========")

for root, dirs, files in os.walk(PROJECT_STORAGE):
    level = root.replace(PROJECT_STORAGE, "").count(os.sep)

    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path)

        print(
            f"{indent}  {file} "
            f"({size / 1024:.2f} KB)"
        )

========== PROJECT STORAGE ==========
AI_Document_Assistant/
  government_documents/
  chroma_db/
    chroma.sqlite3 (184.00 KB)
  personal_documents/


In [84]:
import os

PERSONAL_FOLDER = (
    "/content/drive/MyDrive/"
    "AI_Document_Assistant/personal_documents"
)

print("Personal documents folder exists:",
      os.path.exists(PERSONAL_FOLDER))

print("\nFiles found:")

if os.path.exists(PERSONAL_FOLDER):
    files = os.listdir(PERSONAL_FOLDER)

    if files:
        for file in files:
            print("-", file)
    else:
        print("Folder is empty.")

Personal documents folder exists: True

Files found:
Folder is empty.


In [85]:
import os

print("========== SEARCHING COLAB STORAGE ==========")

for search_folder in ["/content", "/tmp"]:

    print(f"\nSearching: {search_folder}")

    for root, dirs, files in os.walk(search_folder):

        dirs[:] = [
            d for d in dirs
            if d not in ["node_modules", ".cache"]
        ]

        for file in files:
            if (
                file.endswith((".db", ".sqlite", ".sqlite3", ".pdf"))
                and "chroma_db" not in root
            ):
                print(os.path.join(root, file))

========== SEARCHING COLAB STORAGE ==========

Searching: /content
/content/.config/default_configs.db
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db

Searching: /tmp


In [86]:
import os
import sqlite3

PROJECT_STORAGE = (
    "/content/drive/MyDrive/AI_Document_Assistant"
)

PERSONAL_FOLDER = os.path.join(
    PROJECT_STORAGE, "personal_documents"
)

DATABASE_PATH = os.path.join(
    PROJECT_STORAGE, "personal_documents.db"
)

os.makedirs(PERSONAL_FOLDER, exist_ok=True)

connection = sqlite3.connect(DATABASE_PATH)

cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS personal_documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document TEXT,
    type TEXT,
    issue_date TEXT,
    expiry_date TEXT,
    status TEXT,
    confidence REAL,
    file_path TEXT
)
""")

connection.commit()
connection.close()

print("Persistent storage initialized!")
print("Database:", DATABASE_PATH)
print("Personal folder:", PERSONAL_FOLDER)
print("Database exists:", os.path.exists(DATABASE_PATH))

Persistent storage initialized!
Database: /content/drive/MyDrive/AI_Document_Assistant/personal_documents.db
Personal folder: /content/drive/MyDrive/AI_Document_Assistant/personal_documents
Database exists: True


In [87]:
import sqlite3
import os

def load_documents_from_database():

    connection = sqlite3.connect(DATABASE_PATH)
    connection.row_factory = sqlite3.Row

    cursor = connection.cursor()

    cursor.execute("""
        SELECT * FROM personal_documents
        ORDER BY id
    """)

    records = cursor.fetchall()

    documents = [dict(row) for row in records]

    connection.close()

    return documents


def save_document_to_database(doc):

    connection = sqlite3.connect(DATABASE_PATH)
    cursor = connection.cursor()

    cursor.execute("""
        INSERT INTO personal_documents
        (document, type, issue_date, expiry_date,
         status, confidence, file_path)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        doc.get("document", doc.get("file_name", "")),
        doc.get("type", doc.get("document_type", "Unknown")),
        doc.get("issue_date", ""),
        doc.get("expiry_date", ""),
        doc.get("status", "Unknown"),
        doc.get("confidence", 0),
        doc.get("file_path", "")
    ))

    connection.commit()
    connection.close()


# Restore global documents variable
documents = load_documents_from_database()

print("Database functions initialized!")
print("Total personal documents:", len(documents))
print("Documents:", documents)

Database functions initialized!
Total personal documents: 0
Documents: []


In [88]:
import inspect

print("process_personal_document exists:",
      "process_personal_document" in globals())

if "process_personal_document" in globals():
    print(inspect.getsource(process_personal_document))
else:
    print("Function is not currently defined.")

process_personal_document exists: True
def process_personal_document(file_path, documents):

    # Extract text from uploaded PDF/image
    extracted_text = extract_document_text(file_path)

    if not extracted_text or not extracted_text.strip():
        raise ValueError("Could not extract text from the uploaded document.")

    # Extract details using Navigate Labs
    details = extract_document_details(extracted_text)

    if not details:
        raise ValueError("AI document extraction failed.")

    # Calculate document status
    expiry_date = details.get("expiry_date", "")

    if expiry_date:
        try:
            expiry = datetime.strptime(expiry_date, "%Y-%m-%d").date()
            today = date.today()

            if expiry < today:
                status = "Expired"
            elif (expiry - today).days <= 30:
                status = "Expiring Soon"
            else:
                status = "Valid"

        except ValueError:
            status = "Unknown"
    else:
 

In [98]:
print("Documents variable type:", type(documents))
print("Number of records:", len(documents))

for i, doc in enumerate(documents):
    print(f"\nRecord {i}:")
    print("Type:", type(doc))
    print("Value:", doc)

Documents variable type: <class 'list'>
Number of records: 1

Record 0:
Type: <class 'dict'>
Value: {'id': 1, 'document': 'sample_certificate.pdf', 'type': 'Training Certificate', 'issue_date': '2026-11-01', 'expiry_date': '2027-11-01', 'status': 'Valid', 'confidence': '', 'file_path': '/content/drive/MyDrive/AI_Document_Assistant/personal_documents/sample_certificate.pdf'}


In [100]:
print("Database path:", DATABASE_PATH)
print("Database exists:", os.path.exists(DATABASE_PATH))

print("\nDatabase functions available:")

# Create a copy of the keys to avoid changing size during iteration
for name in list(globals()):
    if any(word in name.lower() for word in [
        "database", "load_document", "get_documents"
    ]):
        print(name)

Database path: /content/drive/MyDrive/AI_Document_Assistant/personal_documents.db
Database exists: True

Database functions available:
DATABASE_PATH
load_documents_from_database
save_document_to_database


In [101]:
import inspect

print("LOAD FUNCTION:")
print(inspect.getsource(load_documents_from_database))

print("\nFUNCTION OUTPUT:")
loaded_docs = load_documents_from_database()
print("Type:", type(loaded_docs))
print("Number of records:", len(loaded_docs))
print("First record:", loaded_docs[0] if loaded_docs else "No records found")

LOAD FUNCTION:
def load_documents_from_database():

    connection = sqlite3.connect(DATABASE_PATH)
    connection.row_factory = sqlite3.Row

    cursor = connection.cursor()

    cursor.execute("""
        SELECT * FROM personal_documents
        ORDER BY id
    """)

    records = cursor.fetchall()

    documents = [dict(row) for row in records]

    connection.close()

    return documents


FUNCTION OUTPUT:
Type: <class 'list'>
Number of records: 1
First record: {'id': 1, 'document': 'sample_certificate.pdf', 'type': 'Training Certificate', 'issue_date': '2026-11-01', 'expiry_date': '2027-11-01', 'status': 'Valid', 'confidence': '', 'file_path': '/content/drive/MyDrive/AI_Document_Assistant/personal_documents/sample_certificate.pdf'}


In [102]:
# Restore personal documents from SQLite
documents = load_documents_from_database()

print("Personal documents restored:", len(documents))

for doc in documents:
    print("\nDocument:", doc["document"])
    print("Type:", doc["type"])
    print("Expiry:", doc["expiry_date"])
    print("Status:", doc["status"])

print("\nVariable type:", type(documents))
print("Record type:", type(documents[0]))

Personal documents restored: 1

Document: sample_certificate.pdf
Type: Training Certificate
Expiry: 2027-11-01
Status: Valid

Variable type: <class 'list'>
Record type: <class 'dict'>


In [103]:
test_questions = [
    "Show my uploaded documents",
    "What is the expiry status of my documents?",
    "How can I renew my passport?",
    "What documents are accepted as Aadhaar proof of identity?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}: {question}")
    print("="*60)

    try:
        response = agent_assistant(question, documents)
        print(response)
    except Exception as e:
        print("ERROR:", type(e).__name__, str(e))


TEST 1: Show my uploaded documents
ERROR: NameError name 'personal_document_tool' is not defined

TEST 2: What is the expiry status of my documents?
ERROR: NameError name 'personal_document_tool' is not defined

TEST 3: How can I renew my passport?
ERROR: NameError name 'personal_document_tool' is not defined

TEST 4: What documents are accepted as Aadhaar proof of identity?
ERROR: NameError name 'government_rag_tool' is not defined


In [104]:
def agent_assistant(question, documents):

    question_lower = question.lower()

    # 1. Renewal guidance
    if any(word in question_lower for word in [
        "renew", "renewal", "reissue"
    ]):
        personal_result = personal_document_tool(question, documents)
        government_result = renewal_guidance_tool(question)

        tool_result = (
            "PERSONAL DOCUMENT INFORMATION:\n" + str(personal_result)
            + "\n\nGOVERNMENT RENEWAL GUIDANCE:\n" + str(government_result)
        )
        selected_tool = "Renewal Guidance Tool"

    # 2. Expiry tracking - check before personal documents
    elif any(word in question_lower for word in [
        "expiry", "expire", "expired", "validity", "days remaining"
    ]):
        tool_result = expiry_tracking_tool(question, documents)
        selected_tool = "Expiry Tracking Tool"

    # 3. Personal documents
    elif any(word in question_lower for word in [
        "uploaded documents", "my documents",
        "personal documents", "show my documents",
        "list my documents"
    ]):
        tool_result = personal_document_tool(question, documents)
        selected_tool = "Personal Document Tool"

    # 4. Government RAG
    else:
        tool_result = government_rag_tool(question)
        selected_tool = "Government RAG Tool"

    prompt = f"""
You are an AI Personal Document and Government Service Assistant.

User Question:
{question}

Selected Tool:
{selected_tool}

Retrieved Information:
{tool_result}

Instructions:
- Answer clearly and simply.
- Use only retrieved information.
- Never invent expiry dates or renewal procedures.
- If information is unavailable, say so.
- Preserve government PDF source references.
"""

    answer = nexus_generate(prompt)

    return f"🤖 Selected Tool: {selected_tool}\n\n{answer}"

In [107]:
from datetime import datetime, date

# Fallback definition for check_expiry
if 'check_expiry' not in globals():
    def check_expiry(documents):
        results = []
        today = date.today()
        for doc in documents:
            expiry_date = doc.get("expiry_date")
            if not expiry_date:
                status = "Not applicable"
                days_remaining = "N/A"
            else:
                try:
                    expiry = datetime.strptime(str(expiry_date), "%Y-%m-%d").date()
                    days = (expiry - today).days
                    days_remaining = days
                    if days < 0:
                        status = "Expired"
                    elif days <= 30:
                        status = "Expiring Soon"
                    else:
                        status = "Valid"
                except (ValueError, TypeError):
                    status = "Invalid Date"
                    days_remaining = "N/A"
            results.append({
                "document": doc.get("document", "Unknown"),
                "type": doc.get("type", "Unknown"),
                "issue_date": doc.get("issue_date") or "Not found",
                "expiry_date": expiry_date or "Not applicable",
                "days_remaining": days_remaining,
                "status": status
            })
        return results

# Inline fallback definitions for tools if they aren't loaded
if 'expiry_tracking_tool' not in globals():
    def expiry_tracking_tool(question, documents):
        if not documents:
            return "No personal documents are available for expiry checking."
        expiry_results = check_expiry(documents)
        result = []
        for doc in expiry_results:
            result.append(
                f"Document: {doc.get('document', 'Unknown')}\n"
                f"Expiry Date: {doc.get('expiry_date', 'Not available')}\n"
                f"Days Remaining: {doc.get('days_remaining', 'Not available')}\n"
                f"Status: {doc.get('status', 'Unknown')}\n"
            )
        return "\n".join(result)

print(agent_assistant(
    "What is the expiry status of my documents?",
    documents
))

🤖 Selected Tool: Expiry Tracking Tool

Your document, sample_certificate.pdf, is valid and does not expire until November 1, 2027. There are 347 days remaining before it expires.


In [108]:
import inspect

print("Checking active functions for Gemini references...\n")

function_names = [
    "agent_assistant",
    "rag_answer",
    "extract_document_details",
    "process_personal_document",
    "retrieve_information",
    "plan_tasks"
]

for name in function_names:
    if name in globals() and callable(globals()[name]):
        source = inspect.getsource(globals()[name])
        print(f"{name}:")
        print("Gemini reference:", "gemini" in source.lower())
        print("-" * 40)

Checking active functions for Gemini references...

agent_assistant:
Gemini reference: False
----------------------------------------
rag_answer:
Gemini reference: False
----------------------------------------
extract_document_details:
Gemini reference: False
----------------------------------------
process_personal_document:
Gemini reference: False
----------------------------------------
retrieve_information:
Gemini reference: False
----------------------------------------


In [187]:
import inspect

print("PLAN_TASKS FUNCTION:\n")
print(inspect.getsource(plan_tasks))

print("\nChecking references in the notebook:")

for name, obj in globals().copy().items():
    if callable(obj) and name != "plan_tasks":
        try:
            source = inspect.getsource(obj)
            if "plan_tasks(" in source:
                print("Referenced by:", name)
        except (OSError, TypeError):
            pass

PLAN_TASKS FUNCTION:

def plan_tasks(question):

    prompt = f"""
You are an AI task planner.

Available tools:
1. search_guidelines
2. check_documents

Select the tools required to answer the user's question.

Return JSON in this format:
{{
    "tools": ["search_guidelines", "check_documents"],
    "search_query": "Relevant government information"
}}

Choose only useful tools.

User question:
{question}
"""

    try:
        result = nexus_generate(prompt, json_mode=True)
        plan = json.loads(result)

        allowed_tools = {
            "search_guidelines",
            "check_documents"
        }

        selected = [
            tool for tool in plan.get("tools", [])
            if tool in allowed_tools
        ]

        if not selected:
            selected = ["search_guidelines"]

        return {
            "tools": selected,
            "search_query": plan.get("search_query", question)
        }

    except Exception as e:
        print("Task planner fallback:", str(e)

In [110]:
print("========== FINAL API AUDIT ==========")

gemini_references = []

for name, obj in globals().copy().items():
    if callable(obj):
        try:
            source = inspect.getsource(obj)
            if "gemini_generate" in source.lower() or "generativelanguage.googleapis.com" in source.lower():
                gemini_references.append(name)
        except (OSError, TypeError):
            pass

print("Functions containing Gemini references:", gemini_references)

print("\nNavigate Labs functions:")
print("nexus_generate:", callable(globals().get("nexus_generate")))
print("nexus_embedding:", callable(globals().get("nexus_embedding")))

print("\nKnowledge base chunks:", nexus_collection.count())
print("Personal document records:", len(documents))

========== FINAL API AUDIT ==========
Functions containing Gemini references: []

Navigate Labs functions:
nexus_generate: True
nexus_embedding: True

Knowledge base chunks: 0
Personal document records: 1


In [111]:
for name in ["gemini_generate", "generate_embedding", "func"]:
    obj = globals().get(name)

    print(f"\n{'='*50}")
    print("FUNCTION:", name)

    if callable(obj):
        try:
            print(inspect.getsource(obj))
        except Exception as e:
            print("Source unavailable:", e)
    else:
        print("Not callable")


FUNCTION: gemini_generate
Not callable

FUNCTION: generate_embedding
Not callable

FUNCTION: func
Not callable


In [112]:
# Redirect old Gemini function names to Navigate Labs

def gemini_generate(prompt, json_mode=False):
    return nexus_generate(prompt, json_mode=json_mode)


def generate_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    return nexus_embedding(text)


# Replace the old assistant alias
func = agent_assistant

print("Legacy function compatibility updated!")
print("gemini_generate → Navigate Labs")
print("generate_embedding → Navigate Labs")
print("func → Current agent_assistant")

Legacy function compatibility updated!
gemini_generate → Navigate Labs
generate_embedding → Navigate Labs
func → Current agent_assistant


In [188]:
print("CHAT TEST:")
print(gemini_generate("Reply only: Navigate Labs is connected."))

print("\nEMBEDDING TEST:")
embedding = generate_embedding("Government document assistant")
print("Dimensions:", len(embedding))

print("\nASSISTANT TEST:")
print(agent_assistant("Show my uploaded documents", documents))

CHAT TEST:
Navigate Labs is connected.

EMBEDDING TEST:
Dimensions: 768

ASSISTANT TEST:
🤖 Selected Tool: Personal Document Tool

Here are your uploaded documents based on your personal records:

* **Document Name:** sample_certificate.pdf
* **Type:** Training Certificate
* **Issue Date:** 2026-11-01
* **Expiry Date:** 2027-11-01
* **Status:** Valid

*Note: Information regarding renewal procedures is unavailable.*


In [114]:
!apt-get -qq update
!apt-get -qq install tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [115]:
!pip install -q chromadb pymupdf pytesseract
import chromadb
import pymupdf
import pytesseract
import gradio
import pandas
import requests

print("All project libraries imported successfully!")

All project libraries imported successfully!


In [116]:
import requests
import json
import re
import uuid
import os
import time

from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

MODEL_NAME = "gemini-3.5-flash-lite"
EMBED_MODEL = "gemini-embedding-001"

print("Gemini API connected successfully!")

Gemini API connected successfully!


In [117]:
import requests
import time

def gemini_generate(prompt, json_mode=False):

    url = (
        f"https://generativelanguage.googleapis.com/v1beta/"
        f"models/{MODEL_NAME}:generateContent"
    )

    payload = {
        "contents": [
            {"parts": [{"text": prompt}]}
        ],
        "generationConfig": {
            "temperature": 0.2,
            "maxOutputTokens": 2048
        }
    }

    if json_mode:
        payload["generationConfig"]["responseMimeType"] = "application/json"

    for attempt in range(2):

        try:
            response = requests.post(
                url,
                headers={"x-goog-api-key": API_KEY},
                json=payload,
                timeout=(15, 45)
            )

            if response.status_code == 200:
                result = response.json()
                return result["candidates"][0]["content"]["parts"][0]["text"]

            if response.status_code in [503, 429]:
                print("Gemini temporarily unavailable.")
                time.sleep(5)
                continue

            print("API Error:", response.text[:500])
            response.raise_for_status()

        except requests.exceptions.Timeout:
            print("Gemini request timed out.")

        except requests.exceptions.RequestException as e:
            print("Connection error:", str(e))

    raise Exception(
        "Gemini did not respond. Please try again later."
    )

In [118]:
print("Testing Gemini...")

try:
    result = gemini_generate(
        "Reply with exactly: Gemini is working."
    )
    print("SUCCESS:", result)

except Exception as e:
    print("FAILED:", e)

Testing Gemini...
SUCCESS: Gemini is working.


In [119]:
!pip install -q chromadb

import chromadb

chroma_client = chromadb.PersistentClient(
    path="/content/chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="government_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB initialized successfully!")

ChromaDB initialized successfully!


In [120]:
import time
import requests

def generate_embedding(text, task_type="RETRIEVAL_DOCUMENT"):

    url = (
        "https://generativelanguage.googleapis.com/v1beta/"
        "models/gemini-embedding-001:embedContent"
    )

    payload = {
        "model": "models/gemini-embedding-001",
        "content": {
            "parts": [{"text": text}]
        },
        "taskType": task_type,
        "outputDimensionality": 768
    }

    for attempt in range(5):

        response = requests.post(
            url,
            headers={"x-goog-api-key": API_KEY},
            json=payload,
            timeout=60
        )

        if response.status_code == 200:
            return response.json()["embedding"]["values"]

        if response.status_code == 429:

            wait_time = min(60, 5 * (2 ** attempt))

            print(
                f"Rate limit reached. "
                f"Retrying in {wait_time} seconds..."
            )

            time.sleep(wait_time)

        else:
            print("API Error:", response.text[:500])
            response.raise_for_status()

    raise Exception(
        "Gemini embedding quota exceeded. "
        "Please check your API usage and try again later."
    )

In [122]:
def split_text(text, chunk_size=1000, overlap=150):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [121]:
def add_to_knowledge_base(files):

    total_chunks = 0

    for file_path in files:

        pages = extract_pdf_pages(file_path)

        for page in pages:

            chunks = split_text(page["text"])

            for chunk in chunks:

                # Generate Navigate Labs embedding (1536 dimensions)
                embedding = nexus_embedding(chunk)

                # Store in the new Nexus collection
                nexus_collection.add(
                    ids=[str(uuid.uuid4())],
                    documents=[chunk],
                    embeddings=[embedding],
                    metadatas=[{
                        "source": os.path.basename(file_path),
                        "page": page["page"]
                    }]
                )

                total_chunks += 1

                time.sleep(2)

    return f"Successfully indexed {total_chunks} text chunks."

In [126]:
import os

# Define the folder if it isn't already set in the globals
if 'GOVERNMENT_DOCS_FOLDER' not in globals():
    GOVERNMENT_DOCS_FOLDER = "/content/drive/MyDrive/AI_Document_Assistant/government_documents"

print("Government documents folder:", GOVERNMENT_DOCS_FOLDER)

if os.path.exists(GOVERNMENT_DOCS_FOLDER):
    for filename in os.listdir(GOVERNMENT_DOCS_FOLDER):
        if "ApplicationformInstructionBooklet" in filename:
            print("Found:", filename)
else:
    print("Folder does not exist yet. Please make sure your Google Drive is mounted.")

Government documents folder: /content/drive/MyDrive/AI_Document_Assistant/government_documents


In [128]:
import os
import pymupdf

# Fallback definition for extract_pdf_pages if not defined
if 'extract_pdf_pages' not in globals():
    def extract_pdf_pages(pdf_path):
        pages = []
        with pymupdf.open(pdf_path) as pdf:
            for page_number, page in enumerate(pdf, start=1):
                text = page.get_text("text")
                if text.strip():
                    pages.append({
                        "page": page_number,
                        "text": text
                    })
        return pages

pdf_path = os.path.join(
    GOVERNMENT_DOCS_FOLDER,
    "ApplicationformInstructionBooklet-V3.0.pdf"
)

print("PDF exists:", os.path.exists(pdf_path))

# Check whether this PDF is already indexed
existing = nexus_collection.get(
    where={"source": "ApplicationformInstructionBooklet-V3.0.pdf"},
    include=["metadatas"]
)

print("Existing indexed chunks:", len(existing["ids"]))

# Check PDF content
if os.path.exists(pdf_path):
    pages = extract_pdf_pages(pdf_path)
    print("Total PDF pages:", len(pages))
    if pages:
        print("First page text available:", bool(pages[0]["text"].strip()))
    else:
        print("No readable text pages found.")
else:
    print("PDF file does not exist at path.")

PDF exists: False
Existing indexed chunks: 0
PDF file does not exist at path.


In [130]:
def add_to_knowledge_base(files):

    total_chunks = 0
    skipped_files = []

    for file_path in files:

        filename = os.path.basename(file_path)

        existing = nexus_collection.get(
            where={"source": filename},
            include=["metadatas"]
        )

        if existing["ids"]:
            skipped_files.append(filename)
            continue

        pages = extract_pdf_pages(file_path)

        for page in pages:

            chunks = split_text(page["text"])

            for chunk in chunks:

                embedding = nexus_embedding(chunk)

                nexus_collection.add(
                    ids=[str(uuid.uuid4())],
                    documents=[chunk],
                    embeddings=[embedding],
                    metadatas=[{
                        "source": filename,
                        "page": page["page"]
                    }]
                )

                total_chunks += 1
                time.sleep(2)

    message = f"Successfully indexed {total_chunks} text chunks."

    if skipped_files:
        message += "\nAlready indexed (skipped): " + ", ".join(skipped_files)

    return message

In [131]:
def retrieve_information(question):

    if collection.count() == 0:
        return []

    query_embedding = generate_embedding(
        question,
        task_type="RETRIEVAL_QUERY"
    )

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(4, collection.count())
    )

    retrieved = []

    for i in range(len(results["documents"][0])):

        retrieved.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "page": results["metadatas"][0][i]["page"]
        })

    return retrieved

In [134]:
import inspect

for i, block in demo.fns.items():
    fn = block.fn
    try:
        source = inspect.getsource(fn)
        gemini_url = "generativelanguage.googleapis.com" in source
    except:
        gemini_url = "Unable to inspect"

    print(
        f"Callback {i}: {fn.__name__} | "
        f"Direct Gemini API: {gemini_url}"
    )

In [135]:
def rag_answer(question):

    documents = retrieve_information(question)

    if not documents:

        return "Please upload official government guidelines first."

    context = ""

    for doc in documents:

        context += f"""
Source: {doc['source']}
Page: {doc['page']}

Content:
{doc['text']}

-----------------------
"""

    prompt = f"""
You are an accurate Government Service Information Assistant.

Answer the user's question using ONLY the retrieved government documents.

Instructions:

1. Understand exactly what the user is asking.
2. Identify whether the question concerns a fresh application,
   renewal, reissue, eligibility or fees.
3. Separate general requirements from special-case requirements.
4. Do not mix requirements for different applicant categories.
5. Do not assume that every listed document is mandatory.
6. If the question concerns renewal, prioritize renewal-related information.
7. If the retrieved context does not contain sufficient information,
   clearly say so.
8. Never invent government rules, fees or procedures.
9. Mention the source filename and page number.
10. Explain everything in simple language.

Use this response structure:

- Direct Answer
- General Requirements
- Special Cases (if applicable)
- Important Notes
- Sources

Retrieved Information:
{context}

User Question:
{question}
"""

    return gemini_generate(prompt)

In [136]:
import os
import pymupdf
import pytesseract
from PIL import Image

def extract_document_text(file_path):

    extension = os.path.splitext(file_path)[1].lower()

    # Handle PDF files
    if extension == ".pdf":

        doc = pymupdf.open(file_path)
        text = ""

        for page in doc:

            page_text = page.get_text()

            # OCR fallback for scanned PDF pages
            if not page_text.strip():

                pix = page.get_pixmap()

                image = Image.frombytes(
                    "RGB",
                    [pix.width, pix.height],
                    pix.samples
                )

                page_text = pytesseract.image_to_string(image)

            text += page_text + "\n"

        doc.close()
        return text

    # Handle image files
    elif extension in [".png", ".jpg", ".jpeg", ".webp"]:

        image = Image.open(file_path)
        return pytesseract.image_to_string(image)

    else:
        raise ValueError("Unsupported file format. Upload a PDF or image.")

In [137]:
def extract_document_details(text):

    prompt = f"""
Analyze the following document text.

Extract these details:

1. Document type
2. Issue date
3. Expiry date
4. Confidence level

Return JSON only.

Use YYYY-MM-DD for dates.

If a date is not explicitly available, return null.
Never invent an expiry date.

Document text:

{text[:10000]}
"""

    result = gemini_generate(prompt, json_mode=True)

    return json.loads(result)

In [138]:
from datetime import datetime, date

def process_personal_document(file_path, documents):

    text = extract_document_text(file_path)

    details = extract_document_details(text)

    record = {
        "document": os.path.basename(file_path),
        "type": details.get("document_type", "Unknown"),
        "issue_date": details.get("issue_date"),
        "expiry_date": details.get("expiry_date"),
        "confidence": details.get("confidence", "Not specified")
    }

    documents.append(record)

    return documents

In [139]:
def check_expiry(documents):

    results = []

    today = date.today()

    for doc in documents:

        expiry = doc.get("expiry_date")

        if not expiry:

            status = "Expiry date unavailable"

        else:

            try:

                expiry_date = datetime.strptime(
                    expiry, "%Y-%m-%d"
                ).date()

                days_left = (expiry_date - today).days

                if days_left < 0:
                    status = "Expired"

                elif days_left <= 30:
                    status = "Expiring Soon"

                else:
                    status = "Active"

            except ValueError:

                status = "Date needs verification"

        results.append({
            "Document": doc["document"],
            "Type": doc["type"],
            "Expiry Date": expiry or "Not found",
            "Status": status
        })

    return results

In [140]:
def plan_tasks(question):

    prompt = f"""
You are an AI task planner.

Available tools:
1. search_guidelines
2. check_documents

Select the tools required to answer the user's question.

Return JSON in this format:
{{
    "tools": ["search_guidelines", "check_documents"],
    "search_query": "Relevant government information"
}}

Choose only useful tools.

User question:
{question}
"""

    try:
        result = nexus_generate(prompt, json_mode=True)
        plan = json.loads(result)

        allowed_tools = {
            "search_guidelines",
            "check_documents"
        }

        selected = [
            tool for tool in plan.get("tools", [])
            if tool in allowed_tools
        ]

        if not selected:
            selected = ["search_guidelines"]

        return {
            "tools": selected,
            "search_query": plan.get("search_query", question)
        }

    except Exception as e:
        print("Task planner fallback:", str(e))

        return {
            "tools": ["search_guidelines", "check_documents"],
            "search_query": question
        }

In [141]:
print(plan_tasks("How can I renew my passport?"))

{'tools': ['search_guidelines', 'check_documents'], 'search_query': 'passport renewal process'}


In [142]:
def agent_assistant(question, documents):

    plan = plan_tasks(question)

    tool_results = {}

    for tool in plan["tools"]:

        if tool == "search_guidelines":

            tool_results["government_information"] = (
                rag_answer(plan["search_query"])
            )

        elif tool == "check_documents":

            tool_results["personal_documents"] = (
                check_expiry(documents)
            )

    prompt = f"""
You are an intelligent personal document
and government service assistant.

User Question:
{question}

Information retrieved by the tools:

{json.dumps(tool_results, indent=2)}

Provide a clear and personalized response.

Rules:
- Do not invent missing information.
- Mention missing documents.
- Do not guarantee government scheme eligibility.
- Distinguish verified information from suggestions.
- Recommend checking the official application portal.
"""

    return gemini_generate(prompt)

In [143]:
import os
import json
import re
from datetime import datetime, date


def process_personal_document(file_path, documents):

    extracted_text = extract_document_text(file_path)

    if not extracted_text.strip():
        raise ValueError("No readable text found.")

    prompt = f"""
    Extract the following information from this document.

    Document text:
    {extracted_text}

    Return valid JSON only:
    {{
        "document_type": "",
        "issue_date": "",
        "expiry_date": "",
        "confidence": ""
    }}

    Use YYYY-MM-DD for dates.
    If a date is unavailable, return null.
    """

    response = gemini_generate(prompt, json_mode=True)
    response = response.strip()
    response = re.sub(r"```json|```", "", response).strip()

    details = json.loads(response)

    expiry_date = details.get("expiry_date")

    status = "Not applicable"

    if expiry_date:
        expiry = datetime.strptime(
            expiry_date, "%Y-%m-%d"
        ).date()

        status = "Expired" if expiry < date.today() else "Valid"

    document_record = {
        "document": os.path.basename(file_path),
        "type": details.get("document_type") or "Unknown",
        "issue_date": details.get("issue_date"),
        "expiry_date": expiry_date,
        "status": status,
        "confidence": details.get("confidence") or "Low"
    }

    # Add to current session
    documents.append(document_record)

    # Save permanently in SQLite
    save_document_to_database(
        document_record,
        file_path
    )

    return documents

In [144]:
from datetime import datetime, date

def check_expiry(documents):

    results = []

    today = date.today()

    for doc in documents:

        expiry_date = doc.get("expiry_date")

        if not expiry_date:

            status = "Not applicable"
            days_remaining = "N/A"

        else:

            try:
                expiry = datetime.strptime(
                    expiry_date, "%Y-%m-%d"
                ).date()

                days = (expiry - today).days
                days_remaining = days

                if days < 0:
                    status = "Expired"

                elif days <= 30:
                    status = "Expiring Soon"

                else:
                    status = "Valid"

            except (ValueError, TypeError):
                status = "Invalid Date"
                days_remaining = "N/A"

        results.append({
            "document": doc.get("document", "Unknown"),
            "type": doc.get("type", "Unknown"),
            "issue_date": doc.get("issue_date") or "Not found",
            "expiry_date": expiry_date or "Not applicable",
            "days_remaining": days_remaining,
            "status": status
        })

    return results

In [145]:
def agent_assistant(question, documents):
    if not question.strip():
        return "Please enter your question."

    try:
        # Search the government knowledge base
        answer = rag_answer(question)
        return answer

    except Exception as e:
        return f"Unable to process your question: {str(e)}"

In [146]:
def agent_assistant(question, documents):
    if not question.strip():
        return "Please enter your question."

    try:
        # Search the government knowledge base
        answer = rag_answer(question)
        return answer

    except Exception as e:
        return f"Unable to process your question: {str(e)}"

In [147]:
def generate_dashboard(documents):

    results = check_expiry(documents or [])

    total = len(results)
    expired = sum(d["status"] == "Expired" for d in results)
    expiring = sum(d["status"] == "Expiring Soon" for d in results)
    valid = sum(d["status"] == "Valid" for d in results)
    no_expiry = sum(d["status"] == "Not applicable" for d in results)

    # Expiry alerts
    alerts = ""

    for doc in results:
        days = doc["days_remaining"]
        name = doc["document"]

        if days == "N/A":
            continue

        if days < 0:
            message = "🔴 EXPIRED"
            color = "#fce8e6"
        elif days <= 7:
            message = f"🔴 Expires in {days} days"
            color = "#fce8e6"
        elif days <= 30:
            message = f"🟠 Expires in {days} days"
            color = "#fff4df"
        else:
            message = "🟢 No immediate action required"
            color = "#e6f4ea"

        alerts += f"""
        <div style="background:{color};padding:12px;
                    border-radius:8px;margin:8px 0;">
            <b>{name}</b><br>{message}
        </div>
        """

    if not alerts:
        alerts = "<p>🎉 No expiry alerts. Your documents are all clear!</p>"

    # Recent uploads table
    recent_uploads = ""

    for doc in reversed(results[-5:]):
        recent_uploads += f"""
        <tr>
            <td>{doc['document']}</td>
            <td>{doc['type']}</td>
            <td>{doc['issue_date']}</td>
            <td>{doc['expiry_date']}</td>
            <td>{doc['status']}</td>
        </tr>
        """

    if not recent_uploads:
        recent_uploads = """
        <tr><td colspan="5" style="text-align:center;">
        No documents uploaded yet.
        </td></tr>
        """

    return f"""
    <div style="width:100%;padding:20px;">

        <h2>📊 Document Management Dashboard</h2>
        <p>Your personal document management overview</p>

        <div style="display:flex;flex-wrap:wrap;gap:15px;">

            <div style="flex:1;min-width:180px;background:#e8f0fe;padding:20px;border-radius:12px;">
                <h3>📁 Total Documents</h3><h1>{total}</h1>
            </div>

            <div style="flex:1;min-width:180px;background:#e6f4ea;padding:20px;border-radius:12px;">
                <h3>✅ Valid Documents</h3><h1>{valid}</h1>
            </div>

            <div style="flex:1;min-width:180px;background:#fff4df;padding:20px;border-radius:12px;">
                <h3>⚠️ Expiring Soon</h3><h1>{expiring}</h1>
            </div>

            <div style="flex:1;min-width:180px;background:#fce8e6;padding:20px;border-radius:12px;">
                <h3>❌ Expired Documents</h3><h1>{expired}</h1>
            </div>

        </div>

        <br>

        <div style="background:#f3f4f6;padding:15px;border-radius:10px;">
            📄 <b>Documents without expiry date:</b> {no_expiry}
        </div>

        <br>

        <h2>🔔 Document Expiry Alerts</h2>
        {alerts}

        <br>

        <h2>📂 Recent Uploads</h2>

        <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;">
            <thead>
                <tr style="background:#e8f0fe;">
                    <th style="padding:12px;">Document</th>
                    <th style="padding:12px;">Type</th>
                    <th style="padding:12px;">Issue Date</th>
                    <th style="padding:12px;">Expiry Date</th>
                    <th style="padding:12px;">Status</th>
                </tr>
            </thead>
            <tbody>
                {recent_uploads}
            </tbody>
        </table>
        </div>

    </div>
    """

In [148]:
def save_document_edits(edited_table, documents):

    if not documents:
        return "No documents available to edit.", documents, [], generate_dashboard(documents)

    try:
        edited_rows = edited_table.values.tolist() if hasattr(edited_table, "values") else edited_table

        for index, row in enumerate(edited_rows):

            if index >= len(documents):
                break

            document_type = str(row[1]).strip()
            issue_date = str(row[2]).strip()
            expiry_date = str(row[3]).strip()

            documents[index]["type"] = document_type
            documents[index]["issue_date"] = (
                None if issue_date in ["", "Not found", "None"] else issue_date
            )
            documents[index]["expiry_date"] = (
                None if expiry_date in ["", "Not applicable", "None"] else expiry_date
            )

        # Recalculate document statuses
        expiry_results = check_expiry(documents)
        table = format_expiry_table(expiry_results)
        dashboard = generate_dashboard(documents)

        return "✅ Document details updated successfully!", documents, table, dashboard

    except Exception as e:
        return f"❌ Update failed: {str(e)}", documents, format_expiry_table(check_expiry(documents)), generate_dashboard(documents)

In [149]:
def search_documents(search_query, selected_status, documents):

    results = check_expiry(documents or [])

    filtered_results = []

    for doc in results:

        # Search by document name or type
        query = (search_query or "").strip().lower()

        matches_search = (
            query in doc["document"].lower()
            or query in doc["type"].lower()
        )

        # Filter by status
        matches_status = (
            selected_status == "All"
            or doc["status"] == selected_status
        )

        if matches_search and matches_status:
            filtered_results.append(doc)

    return format_expiry_table(filtered_results)

In [150]:
from datetime import date, datetime

reminder_history = []

def generate_reminders(documents):

    global reminder_history

    today = date.today()
    generated = []

    reminder_days = {
        7: "7 Days Before Expiry",
        3: "3 Days Before Expiry",
        0: "On Expiry Date"
    }

    for index, doc in enumerate(documents):

        expiry = doc.get("expiry_date")

        if not expiry:
            continue

        try:
            expiry_date = datetime.strptime(
                expiry, "%Y-%m-%d"
            ).date()

        except ValueError:
            continue

        days_left = (expiry_date - today).days

        if days_left not in reminder_days:
            continue

        reminder_type = reminder_days[days_left]

        reminder_id = (
            f"REM-{index + 1}-{expiry}"
        )

        # Prevent duplicate reminders
        already_exists = any(
            r["Reminder ID"] == reminder_id
            for r in reminder_history
        )

        if already_exists:
            continue

        reminder = {
            "Reminder ID": reminder_id,
            "Document": doc.get("document", "Unknown"),
            "Days Remaining": days_left,
            "Expiry Date": expiry,
            "Reminder Type": reminder_type,
            "Status": "Preview Only",
            "Created At": datetime.now().strftime(
                "%Y-%m-%d %H:%M"
            )
        }

        reminder_history.append(reminder)
        generated.append(reminder)

    return generated


def get_reminder_history():

    return reminder_history

In [193]:
test_documents = [
    {
        "document": "Sample Certificate",
        "type": "Residence Certificate",
        "expiry_date": "2026-11-25"
    }
]

test_results = generate_reminders(test_documents)

print(test_results)

[]


In [194]:
def generate_notification_preview(documents):

    reminders = generate_reminders(documents)

    if not reminders:
        return "No new reminders generated."

    preview = []

    for reminder in reminders:

        message = f"""
        🔔 DOCUMENT EXPIRY ALERT

        Document: {reminder['Document']}

        Expiry Date: {reminder['Expiry Date']}

        Days Remaining: {reminder['Days Remaining']}

        Reminder Type: {reminder['Reminder Type']}

        Status: Preview Only

        Please renew your document before its expiry date.
        """

        preview.append(message)

    return "\n\n".join(preview)

In [195]:
# Reset reminder_history back to a list to resolve the collision with Gradio's Dataframe variable
reminder_history = []

preview = generate_notification_preview(test_documents)
print(preview)

No new reminders generated.


In [196]:
import pandas as pd

def display_reminder_history():

    if not reminder_history:
        return pd.DataFrame(
            columns=[
                "Reminder ID",
                "Document",
                "Expiry Date",
                "Reminder Type",
                "Status",
                "Created At"
            ]
        )

    return pd.DataFrame(reminder_history)[[
        "Reminder ID",
        "Document",
        "Expiry Date",
        "Reminder Type",
        "Status",
        "Created At"
    ]]

In [154]:
display_reminder_history()

,Reminder ID,Document,Expiry Date,Reminder Type,Status,Created At


In [155]:
import pandas as pd
from datetime import date, datetime

# Store generated reminders
reminder_records = []


def generate_reminders(documents):

    global reminder_records

    today = date.today()
    generated = []

    reminder_days = {
        7: "7 Days Before Expiry",
        3: "3 Days Before Expiry",
        0: "On Expiry Date"
    }

    for index, doc in enumerate(documents or []):

        expiry = doc.get("expiry_date")

        if not expiry:
            continue

        try:
            expiry_date = datetime.strptime(
                str(expiry), "%Y-%m-%d"
            ).date()

        except (ValueError, TypeError):
            continue

        days_left = (expiry_date - today).days

        if days_left not in reminder_days:
            continue

        reminder_id = f"REM-{index + 1}-{expiry}"

        duplicate = any(
            r["Reminder ID"] == reminder_id
            for r in reminder_records
        )

        if duplicate:
            continue

        reminder = {
            "Reminder ID": reminder_id,
            "Document": doc.get("document", "Unknown"),
            "Expiry Date": str(expiry),
            "Days Remaining": days_left,
            "Reminder Type": reminder_days[days_left],
            "Status": "Preview Only",
            "Created At": datetime.now().strftime(
                "%Y-%m-%d %H:%M"
            )
        }

        reminder_records.append(reminder)
        save_reminder_records()
        generated.append(reminder)

    return generated


def check_expiry_reminders(documents):

    print("DEBUG: Documents received:", documents)

    if not documents:
        return "No documents received from Gradio State.", []

    for doc in documents:
        print("DEBUG: Document:", doc)
        print("DEBUG: Expiry date:", doc.get("expiry_date"))

    reminders = generate_reminders(documents)

    print("DEBUG: Generated reminders:", reminders)

    if not reminders:
        return (
            "Documents received, but none match the 7, 3 or 0-day reminder conditions.",
            []
        )

    rows = [
        [
            r["Document"],
            r["Days Remaining"],
            r["Reminder Type"],
            r["Status"]
        ]
        for r in reminders
    ]

    return (
        f"Monitoring completed. {len(reminders)} reminder(s) generated.",
        rows
    )


def get_reminder_history():

    columns = [
        "Reminder ID",
        "Document",
        "Expiry Date",
        "Reminder Type",
        "Status",
        "Created At"
    ]

    return pd.DataFrame(
        reminder_records,
        columns=columns
    )


def get_notification_previews():

    if not reminder_records:
        return "No reminders available. Please check document expiry first."

    messages = []

    for r in reminder_records:

        message = f"""
🔔 DOCUMENT EXPIRY ALERT

Document: {r['Document']}
Expiry Date: {r['Expiry Date']}
Days Remaining: {r['Days Remaining']}
Reminder Type: {r['Reminder Type']}

Status: Preview Only

Please renew your document before its expiry date.
"""

        messages.append(message)

    return "\n\n-------------------------\n\n".join(messages)


def preview_send_reminders():

    if not reminder_records:
        return "No reminders available for testing."

    return (
        f"Notification test completed successfully!\n"
        f"Total reminders: {len(reminder_records)}\n"
        f"Status: Preview Only\n"
        f"No emails were sent."
    )

In [156]:
test_documents = [
    {
        "document": "Sample Residence Certificate",
        "type": "Residence Certificate",
        "issue_date": "2026-09-01",
        "expiry_date": "2026-11-25",
        "status": "Valid",
        "confidence": "High"
    }
]

In [199]:
from google.colab import drive
import os

# Safely mount Google Drive, avoiding ValueError if it is already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("Google Drive is already mounted.")

Google Drive is already mounted.


In [158]:
import os
import json

REMINDER_FILE = "/content/drive/MyDrive/AI_Document_Assistant/reminders.json"

os.makedirs(os.path.dirname(REMINDER_FILE), exist_ok=True)

# Restore existing reminders or preserve current records
if os.path.exists(REMINDER_FILE):

    with open(REMINDER_FILE, "r") as f:
        saved_records = json.load(f)

    existing_ids = {
        r["Reminder ID"] for r in saved_records
    }

    for record in reminder_records:
        if record["Reminder ID"] not in existing_ids:
            saved_records.append(record)

    reminder_records = saved_records

else:

    with open(REMINDER_FILE, "w") as f:
        json.dump(reminder_records, f, indent=4)


print("Persistent reminder storage initialized!")
print("Existing reminders:", len(reminder_records))

Persistent reminder storage initialized!
Existing reminders: 0


In [159]:
def save_reminder_records():

    with open(REMINDER_FILE, "w") as f:
        json.dump(reminder_records, f, indent=4)

    return "Reminder records saved successfully!"

In [160]:
print(save_reminder_records())

print("Total saved:", len(reminder_records))

print("File exists:", os.path.exists(REMINDER_FILE))

Reminder records saved successfully!
Total saved: 0
File exists: True


In [161]:
with open(REMINDER_FILE, "r") as f:
    saved_data = json.load(f)

print("Saved reminder data:")
print(saved_data)

print("Total records:", len(saved_data))

Saved reminder data:
[]
Total records: 0


In [162]:
print("Current reminder_records:")
print(reminder_records)

print("Total:", len(reminder_records))

Current reminder_records:
[]
Total: 0


In [163]:
test_documents_all = [
    {
        "document": "Document A",
        "type": "Certificate",
        "expiry_date": "2026-11-25"
    },
    {
        "document": "Document B",
        "type": "Certificate",
        "expiry_date": "2026-11-21"
    },
    {
        "document": "Document C",
        "type": "Certificate",
        "expiry_date": "2026-11-18"
    }
]

results = generate_reminders(test_documents_all)

print("Generated:", len(results))
print("Current records:", len(reminder_records))

Generated: 0
Current records: 0


In [164]:
save_reminder_records()

print("Saved successfully!")
print("File exists:", os.path.exists(REMINDER_FILE))

Saved successfully!
File exists: True


In [165]:
with open(REMINDER_FILE, "r") as f:
    saved_data = json.load(f)

print("Total saved:", len(saved_data))
print(saved_data)

Total saved: 0
[]


In [167]:
import pandas as pd
import json
import os

# Save reminder records to Google Drive
def save_reminder_records():
    with open(REMINDER_FILE, "w") as f:
        json.dump(reminder_records, f, indent=4)

    return "Reminder records saved successfully!"


# Display saved reminder history
def get_reminder_history():

    columns = [
        "Reminder ID",
        "Document",
        "Expiry Date",
        "Reminder Type",
        "Status",
        "Created At"
    ]

    return pd.DataFrame(reminder_records, columns=columns)


# Generate notification previews from saved records
def get_notification_previews():

    if not reminder_records:
        return "No reminders available."

    messages = []

    for r in reminder_records:

        message = f"""
🔔 DOCUMENT EXPIRY ALERT

Document: {r['Document']}
Expiry Date: {r['Expiry Date']}
Days Remaining: {r['Days Remaining']}
Reminder Type: {r['Reminder Type']}

Status: Preview Only

Please renew your document before its expiry date.
"""

        messages.append(message)

    return "\n\n-------------------------\n\n".join(messages)


# Check expiry and save newly generated reminders
def check_expiry_reminders(documents):

    if not documents:
        return "No documents received.", []

    reminders = generate_reminders(documents)

    if reminders:
        save_reminder_records()

    if not reminders:
        return (
            "No new reminders generated. Existing reminders are preserved.",
            []
        )

    rows = [
        [
            r["Document"],
            r["Days Remaining"],
            r["Reminder Type"],
            r["Status"]
        ]
        for r in reminders
    ]

    return (
        f"Monitoring completed. {len(reminders)} reminder(s) generated.",
        rows
    )


print("Reminder functions updated successfully!")
print("Saved reminder count:", len(reminder_records))

Reminder functions updated successfully!
Saved reminder count: 0


In [168]:
# Test restored reminder history
history_df = get_reminder_history()

print("Reminder History:")
display(history_df)

print("Total reminders:", len(history_df))

Reminder History:


,Reminder ID,Document,Expiry Date,Reminder Type,Status,Created At


Total reminders: 0


In [169]:
preview = get_notification_previews()
print(preview)

No reminders available.


In [170]:
required_items = [
    "documents_state",
    "edit_table",
    "reminder_history_table",
    "notification_output",
    "send_button",
    "send_output",
    "check_expiry_reminders",
    "get_reminder_history",
    "get_notification_previews",
    "preview_send_reminders",
    "save_reminder_records"
]

for item in required_items:
    print(
        f"{'✅' if item in globals() else '❌'} {item}"
    )

❌ documents_state
❌ edit_table
❌ reminder_history_table
❌ notification_output
❌ send_button
❌ send_output
✅ check_expiry_reminders
✅ get_reminder_history
✅ get_notification_previews
✅ preview_send_reminders
✅ save_reminder_records


In [171]:
print("Saved reminders:", len(reminder_records))
print("Drive file exists:", os.path.exists(REMINDER_FILE))
print(get_reminder_history())

Saved reminders: 0
Drive file exists: True
Empty DataFrame
Columns: [Reminder ID, Document, Expiry Date, Reminder Type, Status, Created At]
Index: []


In [172]:
import pymupdf

def extract_pdf_pages(pdf_path):
    pages = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text")

            if text.strip():
                pages.append({
                    "page": page_number,
                    "text": text
                })

    return pages

print("PDF page extraction function is ready!")

PDF page extraction function is ready!


In [173]:
# ==========================================
# AGENTIC AI - TOOL DEFINITIONS
# ==========================================

def personal_document_tool(question, documents):
    """Retrieve uploaded personal document information."""

    if not documents:
        return "No personal documents have been uploaded yet."

    result = []

    for doc in documents:
        result.append(
            f"""
            Document: {doc.get('document', 'Unknown')}
            Type: {doc.get('type', 'Unknown')}
            Issue Date: {doc.get('issue_date', 'Not available')}
            Expiry Date: {doc.get('expiry_date', 'Not available')}
            Status: {doc.get('status', 'Unknown')}
            """
        )

    return "\n".join(result)


def expiry_tracking_tool(question, documents):
    """Check expiry information for uploaded documents."""

    if not documents:
        return "No personal documents are available for expiry checking."

    expiry_results = check_expiry(documents)

    if not expiry_results:
        return "No expiry information is available."

    result = []

    for doc in expiry_results:
        result.append(
            f"""
            Document: {doc.get('document', 'Unknown')}
            Expiry Date: {doc.get('expiry_date', 'Not available')}
            Days Remaining: {doc.get('days_remaining', 'Not available')}
            Status: {doc.get('status', 'Unknown')}
            """
        )

    return "\n".join(result)


def government_rag_tool(question):
    """Retrieve information from official government documents."""

    return rag_answer(question)


def renewal_guidance_tool(question):
    """Retrieve government guidance related to renewal."""

    return rag_answer(question)


print("All four agent tools created successfully!")

All four agent tools created successfully!


In [174]:
for name in [
    "personal_document_tool",
    "expiry_tracking_tool",
    "government_rag_tool",
    "renewal_guidance_tool"
]:
    print(f"✅ {name}" if name in globals() else f"❌ {name}")

✅ personal_document_tool
✅ expiry_tracking_tool
✅ government_rag_tool
✅ renewal_guidance_tool


In [175]:
import json
import re

def agent_assistant(question, documents):

    question_lower = question.lower()

    # 1. Personal document questions
    if any(word in question_lower for word in [
        "uploaded documents",
        "my documents",
        "personal documents",
        "show my documents",
        "list my documents"
    ]):
        tool_result = personal_document_tool(question, documents)
        selected_tool = "Personal Document Tool"

    # 2. Renewal questions - PRIORITY
    elif any(word in question_lower for word in [
        "renew",
        "renewal",
        "reissue"
    ]):
        personal_result = personal_document_tool(
            question, documents
        )

        government_result = renewal_guidance_tool(question)

        tool_result = (
            "PERSONAL DOCUMENT INFORMATION:\n"
            + personal_result
            + "\n\nGOVERNMENT RENEWAL GUIDANCE:\n"
            + government_result
        )

        selected_tool = "Renewal Guidance Tool"

    # 3. Expiry questions
    elif any(word in question_lower for word in [
        "expiry",
        "expire",
        "expired",
        "validity",
        "days remaining"
    ]):
        tool_result = expiry_tracking_tool(question, documents)
        selected_tool = "Expiry Tracking Tool"

    # 4. Government information questions
    else:
        tool_result = government_rag_tool(question)
        selected_tool = "Government RAG Tool"

    prompt = f"""
    You are an AI Personal Document and Government Service Assistant.

    User Question:
    {question}

    Selected Tool:
    {selected_tool}

    Retrieved Information:
    {tool_result}

    Instructions:
    - Answer clearly and simply.
    - Use only retrieved information.
    - Never invent expiry dates or renewal procedures.
    - If renewal information is unavailable, say so.
    - Preserve government PDF source references.
    """

    answer = gemini_generate(prompt)

    return (
        f"🤖 Selected Tool: {selected_tool}\n\n"
        f"{answer}"
    )


print("Agent Assistant routing updated successfully!")

Agent Assistant routing updated successfully!


In [176]:
test_documents = [
    {
        "document": "sample_certificate.pdf",
        "type": "Residence Certificate",
        "issue_date": "2026-09-01",
        "expiry_date": "2027-09-01",
        "status": "Valid"
    }
]

print(
    agent_assistant(
        "Show my uploaded personal documents",
        test_documents
    )
)

🤖 Selected Tool: Personal Document Tool

Based on your uploaded documents, here is the information available:

* **Document Name:** sample_certificate.pdf
* **Type:** Residence Certificate
* **Issue Date:** 2026-09-01
* **Expiry Date:** 2027-09-01
* **Status:** Valid

*Note: Information regarding renewal procedures is currently unavailable.*


In [ ]:
test_documents = [
    {
        "document": "sample_certificate.pdf",
        "type": "Residence Certificate",
        "issue_date": "2026-09-01",
        "expiry_date": "2027-09-01",
        "status": "Valid"
    }
]

print(
    agent_assistant(
        "Check the expiry date and tell me how many days are remaining",
        test_documents
    )
)

In [201]:
print(
    agent_assistant(
        "What documents are accepted as Proof of Identity for Aadhaar enrolment?",
        test_documents
    )
)

🤖 Selected Tool: Government RAG Tool

Based on the current government guidelines available, the specific documents accepted as Proof of Identity for Aadhaar enrolment have not been uploaded yet. 

Please upload the official government guidelines first so I can provide you with the accurate list of accepted documents.


In [202]:
print(
    agent_assistant(
        "How can I renew my residence certificate before it expires?",
        test_documents
    )
)

🤖 Selected Tool: Renewal Guidance Tool

Based on the retrieved information, your Residence Certificate (Sample Certificate) is set to expire on **2026-11-25**. 

However, specific renewal procedures and official government guidelines are currently unavailable in the system ("Please upload official government guidelines first"). Therefore, the exact renewal instructions cannot be provided at this time.


In [203]:
import inspect

source = inspect.getsource(agent_assistant)
print("Agent Assistant function is available.")

Agent Assistant function is available.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [205]:
import os

PROJECT_STORAGE = "/content/drive/MyDrive/AI_Document_Assistant"
PERSONAL_DOCS_FOLDER = os.path.join(PROJECT_STORAGE, "personal_documents")

os.makedirs(PERSONAL_DOCS_FOLDER, exist_ok=True)

print("Project storage:", PROJECT_STORAGE)
print("Personal documents folder:", PERSONAL_DOCS_FOLDER)
print("Folder created successfully!")

Project storage: /content/drive/MyDrive/AI_Document_Assistant
Personal documents folder: /content/drive/MyDrive/AI_Document_Assistant/personal_documents
Folder created successfully!


In [206]:
import os
import shutil

def save_personal_file(file_path):

    filename = os.path.basename(file_path)

    destination = os.path.join(
        PERSONAL_DOCS_FOLDER,
        filename
    )

    # Avoid overwriting an existing file
    if os.path.exists(destination):
        name, ext = os.path.splitext(filename)
        counter = 1

        while os.path.exists(destination):
            new_filename = f"{name}_{counter}{ext}"
            destination = os.path.join(
                PERSONAL_DOCS_FOLDER,
                new_filename
            )
            counter += 1

    shutil.copy2(file_path, destination)

    return destination


print("Personal document storage function ready!")

Personal document storage function ready!


In [207]:
import os

saved_files = os.listdir(PERSONAL_DOCS_FOLDER)

print("Total saved files:", len(saved_files))

for filename in saved_files:
    print("📄", filename)

Total saved files: 1
📄 sample_certificate.pdf


In [208]:
import sqlite3
import os

DATABASE_PATH = os.path.join(
    PROJECT_STORAGE,
    "personal_documents.db"
)

def create_database():

    connection = sqlite3.connect(DATABASE_PATH)
    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS personal_documents (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            document TEXT NOT NULL,
            type TEXT,
            issue_date TEXT,
            expiry_date TEXT,
            status TEXT,
            confidence TEXT,
            file_path TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    connection.commit()
    connection.close()

    print("SQLite database created successfully!")
    print("Database location:", DATABASE_PATH)


create_database()

SQLite database created successfully!
Database location: /content/drive/MyDrive/AI_Document_Assistant/personal_documents.db


In [209]:
def save_document_to_database(document_record, file_path):

    connection = sqlite3.connect(DATABASE_PATH)
    cursor = connection.cursor()

    cursor.execute("""
        INSERT INTO personal_documents
        (document, type, issue_date, expiry_date,
         status, confidence, file_path)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        document_record.get("document"),
        document_record.get("type"),
        document_record.get("issue_date"),
        document_record.get("expiry_date"),
        document_record.get("status"),
        document_record.get("confidence"),
        file_path
    ))

    connection.commit()
    connection.close()


print("Document database save function ready!")

Document database save function ready!


In [210]:
connection = sqlite3.connect(DATABASE_PATH)

cursor = connection.cursor()

cursor.execute("""
    SELECT id, document, type, issue_date,
           expiry_date, status, file_path
    FROM personal_documents
""")

records = cursor.fetchall()

print("Total database records:", len(records))

for record in records:
    print("\n", record)

connection.close()

Total database records: 1

 (1, 'sample_certificate.pdf', 'Training Certificate', '2026-11-01', '2027-11-01', 'Valid', '/content/drive/MyDrive/AI_Document_Assistant/personal_documents/sample_certificate.pdf')


In [211]:
def load_documents_from_database():

    connection = sqlite3.connect(DATABASE_PATH)
    connection.row_factory = sqlite3.Row

    cursor = connection.cursor()

    cursor.execute("""
        SELECT document, type, issue_date,
               expiry_date, status, confidence
        FROM personal_documents
        ORDER BY id
    """)

    records = cursor.fetchall()

    documents = [dict(record) for record in records]

    connection.close()

    return documents


print("Document restoration function ready!")

Document restoration function ready!


In [212]:
restored_documents = load_documents_from_database()

for doc in restored_documents:
    print("Document:", doc["document"])
    print("Type:", doc["type"])
    print("Issue Date:", doc["issue_date"])
    print("Expiry Date:", doc["expiry_date"])
    print("Status:", doc["status"])
    print("-" * 40)

Document: sample_certificate.pdf
Type: Training Certificate
Issue Date: 2026-11-01
Expiry Date: 2027-11-01
Status: Valid
----------------------------------------


In [213]:
GOVERNMENT_DOCS_FOLDER = os.path.join(
    PROJECT_STORAGE,
    "government_documents"
)

os.makedirs(GOVERNMENT_DOCS_FOLDER, exist_ok=True)

print("Government PDF folder ready!")
print(GOVERNMENT_DOCS_FOLDER)

Government PDF folder ready!
/content/drive/MyDrive/AI_Document_Assistant/government_documents


In [214]:
def save_government_files(files):

    if not files:
        return "No files uploaded."

    if not isinstance(files, list):
        files = [files]

    saved = []
    failed = []

    for file in files:
        try:
            filename = os.path.basename(file)

            destination = os.path.join(
                GOVERNMENT_DOCS_FOLDER,
                filename
            )

            if os.path.exists(destination):
                name, ext = os.path.splitext(filename)
                counter = 1

                while os.path.exists(destination):
                    destination = os.path.join(
                        GOVERNMENT_DOCS_FOLDER,
                        f"{name}_{counter}{ext}"
                    )
                    counter += 1

            shutil.copy2(file, destination)

            result = add_to_knowledge_base([destination])

            saved.append(
                f"{os.path.basename(destination)}: {result}"
            )

        except Exception as e:
            print("ERROR:", str(e))
            failed.append(f"{filename}: {str(e)}")

    message = f"Successfully processed: {len(saved)} PDF(s)."

    if saved:
        message += "\n\n" + "\n".join(saved)

    if failed:
        message += "\n\nFailed:\n" + "\n".join(failed)

    return message


print("Government PDF upload function ready!")

Government PDF upload function ready!


In [215]:
import inspect

print("Function signature:")
print(inspect.signature(add_to_knowledge_base))

print("\nExisting function code:")
print(inspect.getsource(add_to_knowledge_base))

Function signature:
(files)

Existing function code:
def add_to_knowledge_base(files):

    total_chunks = 0
    skipped_files = []

    for file_path in files:

        filename = os.path.basename(file_path)

        existing = nexus_collection.get(
            where={"source": filename},
            include=["metadatas"]
        )

        if existing["ids"]:
            skipped_files.append(filename)
            continue

        pages = extract_pdf_pages(file_path)

        for page in pages:

            chunks = split_text(page["text"])

            for chunk in chunks:

                embedding = nexus_embedding(chunk)

                nexus_collection.add(
                    ids=[str(uuid.uuid4())],
                    documents=[chunk],
                    embeddings=[embedding],
                    metadatas=[{
                        "source": filename,
                        "page": page["page"]
                    }]
                )

                total_chunks += 1


In [216]:
print("Government folder contents:")

for filename in os.listdir(GOVERNMENT_DOCS_FOLDER):
    print("📄", filename)

Government folder contents:


In [217]:
government_files = [
    os.path.join(GOVERNMENT_DOCS_FOLDER, filename)
    for filename in os.listdir(GOVERNMENT_DOCS_FOLDER)
    if filename.lower().endswith(".pdf")
]

result = add_to_knowledge_base(government_files)

print(result)

Successfully indexed 0 text chunks.


In [218]:
import shutil
import os

OLD_CHROMA_PATH = "/content/chroma_db"

NEW_CHROMA_PATH = os.path.join(
    PROJECT_STORAGE,
    "chroma_db"
)

if os.path.exists(OLD_CHROMA_PATH):

    if not os.path.exists(NEW_CHROMA_PATH):
        shutil.copytree(
            OLD_CHROMA_PATH,
            NEW_CHROMA_PATH
        )
        print("ChromaDB backup copied successfully!")

    else:
        print("Google Drive ChromaDB folder already exists.")

else:
    print("Existing ChromaDB folder not found!")

Google Drive ChromaDB folder already exists.


In [219]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path=NEW_CHROMA_PATH
)

collection = chroma_client.get_or_create_collection(
    name="government_knowledge_clean",
    embedding_function=None
)

print("Persistent ChromaDB initialized successfully!")
print("Total indexed chunks:", collection.count())

Persistent ChromaDB initialized successfully!
Total indexed chunks: 0


In [220]:
print("Persistent database:", NEW_CHROMA_PATH)
print("Total indexed chunks:", collection.count())

print("\nRAG function signature:")
import inspect

print(inspect.signature(retrieve_information))

Persistent database: /content/drive/MyDrive/AI_Document_Assistant/chroma_db
Total indexed chunks: 0

RAG function signature:
(question)


In [221]:
print(inspect.getsource(retrieve_information))

def retrieve_information(question):

    if collection.count() == 0:
        return []

    query_embedding = generate_embedding(
        question,
        task_type="RETRIEVAL_QUERY"
    )

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(4, collection.count())
    )

    retrieved = []

    for i in range(len(results["documents"][0])):

        retrieved.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "page": results["metadatas"][0][i]["page"]
        })

    return retrieved



In [222]:
test_question = "What documents are accepted as proof of address for Aadhaar?"

retrieved_info = retrieve_information(test_question)

print("Retrieved chunks:", len(retrieved_info))

for item in retrieved_info:
    print("\nSource:", item["source"])
    print("Page:", item["page"])
    print("Content:", item["text"][:300])
    print("-" * 50)

Retrieved chunks: 0


In [223]:
from collections import Counter

data = collection.get(
    include=["documents", "metadatas"]
)

print("Total chunks:", len(data["ids"]))

source_counts = Counter(
    metadata["source"]
    for metadata in data["metadatas"]
)

print("\nChunks by source:")

for source, count in source_counts.items():
    print(source, ":", count)

unique_chunks = set(data["documents"])

print("\nUnique text chunks:", len(unique_chunks))
print("Duplicate chunks:", len(data["documents"]) - len(unique_chunks))

Total chunks: 0

Chunks by source:

Unique text chunks: 0
Duplicate chunks: 0


In [224]:
# Retrieve all existing records
data = collection.get(
    include=["documents", "metadatas"]
)

# Identify unique records using source + page + text
unique_records = {}
duplicates = 0

for i, text_content in enumerate(data["documents"]):

    metadata = data["metadatas"][i]

    key = (
        metadata["source"],
        metadata["page"],
        text_content
    )

    if key not in unique_records:
        unique_records[key] = {
            "id": data["ids"][i],
            "document": text_content,
            "metadata": metadata
        }
    else:
        duplicates += 1

print("Original records:", len(data["ids"]))
print("Unique records:", len(unique_records))
print("Duplicates identified:", duplicates)

Original records: 0
Unique records: 0
Duplicates identified: 0


In [ ]:
clean_collection = chroma_client.get_or_create_collection(
    name="government_knowledge_clean",
    metadata={"hnsw:space": "cosine"}
)

# Add unique records only if collection is empty
if clean_collection.count() == 0:

    records = list(unique_records.values())

    clean_collection.add(
        ids=[r["id"] for r in records],
        documents=[r["document"] for r in records],
        metadatas=[r["metadata"] for r in records]
    )

collection = clean_collection

print("Clean knowledge base created!")
print("Clean chunks:", collection.count())

In [ ]:
# Access the original collection (unchanged)
original_collection = chroma_client.get_collection(
    name="government_knowledge"
)

# Retrieve original Gemini embeddings
original_data = original_collection.get(
    include=["documents", "metadatas", "embeddings"]
)

# Rebuild unique records with their original embeddings
unique_records = {}

for i, text_content in enumerate(original_data["documents"]):

    metadata = original_data["metadatas"][i]

    key = (
        metadata["source"],
        metadata["page"],
        text_content
    )

    if key not in unique_records:
        unique_records[key] = {
            "id": original_data["ids"][i],
            "document": text_content,
            "metadata": metadata,
            "embedding": original_data["embeddings"][i].tolist()
        }

# Remove only the temporary clean collection
chroma_client.delete_collection(
    name="government_knowledge_clean"
)

# Recreate without a default embedding function
clean_collection = chroma_client.create_collection(
    name="government_knowledge_clean",
    metadata={"hnsw:space": "cosine"},
    embedding_function=None
)

records = list(unique_records.values())

clean_collection.add(
    ids=[r["id"] for r in records],
    documents=[r["document"] for r in records],
    metadatas=[r["metadata"] for r in records],
    embeddings=[r["embedding"] for r in records]
)

collection = clean_collection

print("Clean Gemini knowledge base created!")
print("Total unique chunks:", collection.count())

In [228]:
test_question = "What documents are accepted as proof of address for Aadhaar?"

retrieved_info = retrieve_information(test_question)

print("Retrieved chunks:", len(retrieved_info))

for item in retrieved_info:
    print("\nSource:", item["source"])
    print("Page:", item["page"])
    print("Content:", item["text"][:250])
    print("-" * 40)

Retrieved chunks: 0


In [227]:
print("PERSONAL DOCUMENTS:")
print(os.listdir(PERSONAL_DOCS_FOLDER))

print("\nGOVERNMENT DOCUMENTS:")
print(os.listdir(GOVERNMENT_DOCS_FOLDER))

print("\nCHROMA DATABASE:")
print(os.listdir(NEW_CHROMA_PATH))

PERSONAL DOCUMENTS:
['sample_certificate.pdf']

GOVERNMENT DOCUMENTS:
[]

CHROMA DATABASE:
['chroma.sqlite3']


In [229]:
# 1. Verify personal documents database
saved_documents = load_documents_from_database()

print("PERSONAL DOCUMENTS IN DATABASE:", len(saved_documents))

for doc in saved_documents:
    print(doc["document"], "|", doc["type"])


# 2. Verify clean ChromaDB collection
chroma_client = chromadb.PersistentClient(
    path=NEW_CHROMA_PATH
)

collection = chroma_client.get_collection(
    name="government_knowledge_clean",
    embedding_function=None
)

print("\nGOVERNMENT KNOWLEDGE BASE")
print("Collection:", collection.name)
print("Total unique chunks:", collection.count())

PERSONAL DOCUMENTS IN DATABASE: 1
sample_certificate.pdf | Training Certificate

GOVERNMENT KNOWLEDGE BASE
Collection: government_knowledge_clean
Total unique chunks: 0


In [230]:
test_question = "What documents are accepted as proof of address for Aadhaar?"

retrieved_info = retrieve_information(test_question)

print("Retrieved chunks:", len(retrieved_info))

for item in retrieved_info:
    print("\nSource:", item["source"])
    print("Page:", item["page"])
    print("Content:", item["text"][:200])

Retrieved chunks: 0


In [231]:
restored_documents = load_documents_from_database()

test_questions = [
    "Show my uploaded personal documents.",
    "What documents are accepted as proof of address for Aadhaar?"
]

for question in test_questions:
    print("\nQUESTION:", question)
    print("-" * 50)

    answer = agent_assistant(question, restored_documents)

    print(answer)
    print("=" * 60)


QUESTION: Show my uploaded personal documents.
--------------------------------------------------
🤖 Selected Tool: Personal Document Tool

Here is your uploaded personal document:

* **Document Name:** sample_certificate.pdf
* **Type:** Training Certificate
* **Issue Date:** 2026-11-01
* **Expiry Date:** 2027-11-01
* **Status:** Valid

*Note: Renewal information is currently unavailable.*

QUESTION: What documents are accepted as proof of address for Aadhaar?
--------------------------------------------------
🤖 Selected Tool: Government RAG Tool

Based on the official government guidelines, the retrieved information states: 

"Please upload official government guidelines first." 

Therefore, the specific list of documents accepted as proof of address for Aadhaar is currently unavailable in the retrieved system data.


In [232]:
print("========== GRADIO APPLICATION CHECK ==========")

import gradio as gr

for name, obj in globals().copy().items():
    if isinstance(obj, gr.Blocks):
        print("Application variable:", name)
        print("Application type:", type(obj).__name__)

print("\n========== API CHECK ==========")

print("Navigate Labs Chat:", callable(nexus_generate))
print("Navigate Labs Embeddings:", callable(nexus_embedding))
print("Knowledge Base:", nexus_collection.count(), "chunks")
print("Personal Documents:", len(documents), "records")

========== GRADIO APPLICATION CHECK ==========
Application variable: demo
Application type: Blocks

========== API CHECK ==========
Navigate Labs Chat: True
Navigate Labs Embeddings: True
Knowledge Base: 0 chunks
Personal Documents: 1 records


In [233]:
print("========== GRADIO CALLBACK AUDIT ==========")

for index, dependency in enumerate(demo.config.get("dependencies", [])):
    fn_index = dependency.get("id", index)

    print(f"\nCallback {index + 1}")
    print("Function index:", fn_index)
    print("Inputs:", dependency.get("inputs"))
    print("Outputs:", dependency.get("outputs"))
    print("API name:", dependency.get("api_name"))

print("\nTotal registered callbacks:",
      len(demo.config.get("dependencies", [])))

========== GRADIO CALLBACK AUDIT ==========

Total registered callbacks: 0


In [234]:
print("========== REGISTERED FUNCTION AUDIT ==========")

for index, block_fn in demo.fns.items():

    fn = getattr(block_fn, "fn", None)

    print(f"\nCallback {index}")
    print("Function:", getattr(fn, "__name__", "Unknown"))

    if callable(fn):
        try:
            source = inspect.getsource(fn)

            print(
                "Gemini reference:",
                "gemini_generate" in source.lower()
                or "generativelanguage.googleapis.com" in source.lower()
            )

            print(
                "Navigate Labs reference:",
                "nexus_generate" in source.lower()
                or "agent_assistant" in source.lower()
            )

        except Exception:
            print("Source inspection unavailable")

========== REGISTERED FUNCTION AUDIT ==========


In [235]:
# Update Gradio's registered AI assistant callback
demo.fns[5].fn = agent_assistant

print("Updated Gradio AI Assistant callback!")

print("Registered function:",
      demo.fns[5].fn.__name__)

print("Uses current assistant:",
      demo.fns[5].fn is agent_assistant)

KeyError: 5

In [236]:
print(demo.fns[5].fn(
    "What is the expiry status of my documents?",
    documents
))

KeyError: 5

In [237]:
import inspect

print("========== UPLOAD FUNCTION ==========")
print(inspect.getsource(upload_personal))

print("\n========== DOCUMENT PROCESSING FUNCTION ==========")
print(inspect.getsource(process_personal_document))

========== UPLOAD FUNCTION ==========
def upload_personal(files, documents):

    if not files:
        return (
            "No files uploaded.",
            documents,
            format_expiry_table(check_expiry(documents)),
            generate_dashboard(documents)
        )

    if not isinstance(files, list):
        files = [files]

    successful = []
    failed = []

    for file in files:
        try:
            # Save original file permanently in Google Drive
            saved_path = save_personal_file(file)

            # Process the saved copy
            documents = process_personal_document(
                saved_path, documents
            )

            successful.append(os.path.basename(saved_path))

        except Exception as e:
            print(f"ERROR processing {file}: {str(e)}")
            failed.append(os.path.basename(file))

    expiry_results = check_expiry(documents)
    table = format_expiry_table(expiry_results)
    dashboard = generate_dashboard(docum

In [238]:
import inspect

print("========== SAVE PERSONAL FILE ==========")
print(inspect.getsource(save_personal_file))

print("\n========== DATABASE SAVE FUNCTION ==========")
print(inspect.getsource(save_document_to_database))

print("\n========== FILE EXTRACTION FUNCTION ==========")
print(inspect.getsource(extract_document_text))

========== SAVE PERSONAL FILE ==========
def save_personal_file(file_path):

    filename = os.path.basename(file_path)

    destination = os.path.join(
        PERSONAL_DOCS_FOLDER,
        filename
    )

    # Avoid overwriting an existing file
    if os.path.exists(destination):
        name, ext = os.path.splitext(filename)
        counter = 1

        while os.path.exists(destination):
            new_filename = f"{name}_{counter}{ext}"
            destination = os.path.join(
                PERSONAL_DOCS_FOLDER,
                new_filename
            )
            counter += 1

    shutil.copy2(file_path, destination)

    return destination


========== DATABASE SAVE FUNCTION ==========
def save_document_to_database(document_record, file_path):

    connection = sqlite3.connect(DATABASE_PATH)
    cursor = connection.cursor()

    cursor.execute("""
        INSERT INTO personal_documents
        (document, type, issue_date, expiry_date,
         status, confidence, file_path)


In [239]:
def process_personal_document(file_path, documents):

    extracted_text = extract_document_text(file_path)

    if not extracted_text or not extracted_text.strip():
        raise ValueError("Could not extract document text.")

    details = extract_document_details(extracted_text)

    if not details:
        raise ValueError("AI document extraction failed.")

    expiry_date = details.get("expiry_date", "")

    if expiry_date:
        try:
            expiry = datetime.strptime(
                expiry_date, "%Y-%m-%d"
            ).date()

            today = date.today()

            if expiry < today:
                status = "Expired"
            elif (expiry - today).days <= 30:
                status = "Expiring Soon"
            else:
                status = "Valid"

        except ValueError:
            status = "Unknown"
    else:
        status = "No Expiry Date"

    document_name = os.path.basename(file_path)
    document_type = details.get("document_type", "Unknown")

    document_record = {
        "document": document_name,
        "type": document_type,
        "file_name": document_name,
        "document_type": document_type,
        "issue_date": details.get("issue_date", ""),
        "expiry_date": expiry_date,
        "confidence": details.get("confidence", ""),
        "status": status
    }

    documents = documents or []
    documents.append(document_record)

    save_document_to_database(document_record, file_path)

    return documents

In [240]:
print("Personal document folder:", PERSONAL_DOCS_FOLDER)

for filename in os.listdir(PERSONAL_DOCS_FOLDER):
    if "fictional sample document" in filename.lower():
        print(filename)

Personal document folder: /content/drive/MyDrive/AI_Document_Assistant/personal_documents


In [241]:
connection = sqlite3.connect(DATABASE_PATH)
cursor = connection.cursor()

cursor.execute("""
    SELECT document, type, file_path
    FROM personal_documents
    WHERE document LIKE '%fictional sample document%'
""")

records = cursor.fetchall()

print("Matching database records:")

for record in records:
    print(record)

connection.close()

Matching database records:


In [242]:
test_file = os.path.join(
    PERSONAL_DOCS_FOLDER,
    "fictional sample document_1.pdf"
)

print("File exists:", os.path.exists(test_file))

try:
    documents = process_personal_document(test_file, documents)

    print("\nDocument processed successfully!")

    print("\nLatest record:")
    print(documents[-1])

except Exception as e:
    print("\nProcessing failed:")
    print(type(e).__name__, str(e))

File exists: False

Processing failed:
FileNotFoundError no such file: '/content/drive/MyDrive/AI_Document_Assistant/personal_documents/fictional sample document_1.pdf'


In [243]:
documents = load_documents_from_database()

print("Total personal documents:", len(documents))

for index, doc in enumerate(documents, 1):
    print(f"\nDocument {index}")
    print("Name:", doc["document"])
    print("Type:", doc["type"])
    print("Issue Date:", doc["issue_date"])
    print("Expiry Date:", doc["expiry_date"])
    print("Status:", doc["status"])

Total personal documents: 1

Document 1
Name: sample_certificate.pdf
Type: Training Certificate
Issue Date: 2026-11-01
Expiry Date: 2027-11-01
Status: Valid


In [244]:
print("Current processing function:")
print(inspect.getsource(process_personal_document))

print("\nCurrent upload callback:")
print(demo.fns[1].fn is upload_personal)

Current processing function:
def process_personal_document(file_path, documents):

    extracted_text = extract_document_text(file_path)

    if not extracted_text or not extracted_text.strip():
        raise ValueError("Could not extract document text.")

    details = extract_document_details(extracted_text)

    if not details:
        raise ValueError("AI document extraction failed.")

    expiry_date = details.get("expiry_date", "")

    if expiry_date:
        try:
            expiry = datetime.strptime(
                expiry_date, "%Y-%m-%d"
            ).date()

            today = date.today()

            if expiry < today:
                status = "Expired"
            elif (expiry - today).days <= 30:
                status = "Expiring Soon"
            else:
                status = "Valid"

        except ValueError:
            status = "Unknown"
    else:
        status = "No Expiry Date"

    document_name = os.path.basename(file_path)
    document_type = details.get(

KeyError: 1

In [246]:
import inspect

print("========== SAVE GOVERNMENT FILES ==========")
print(inspect.getsource(save_government_files))

print("\n========== UPLOAD OFFICIAL ==========")
print(inspect.getsource(upload_official))

print("\n========== COLLECTIONS ==========")
print("Original collection:", collection.name)
print("Original count:", collection.count())

print("Nexus collection:", nexus_collection.name)
print("Nexus count:", nexus_collection.count())

========== SAVE GOVERNMENT FILES ==========
def save_government_files(files):

    if not files:
        return "No files uploaded."

    if not isinstance(files, list):
        files = [files]

    saved = []
    failed = []

    for file in files:
        try:
            filename = os.path.basename(file)

            destination = os.path.join(
                GOVERNMENT_DOCS_FOLDER,
                filename
            )

            if os.path.exists(destination):
                name, ext = os.path.splitext(filename)
                counter = 1

                while os.path.exists(destination):
                    destination = os.path.join(
                        GOVERNMENT_DOCS_FOLDER,
                        f"{name}_{counter}{ext}"
                    )
                    counter += 1

            shutil.copy2(file, destination)

            result = add_to_knowledge_base([destination])

            saved.append(
                f"{os.path.basename(destination)}: {result}"

In [245]:
import inspect

print(inspect.getsource(add_to_knowledge_base))

def add_to_knowledge_base(files):

    total_chunks = 0
    skipped_files = []

    for file_path in files:

        filename = os.path.basename(file_path)

        existing = nexus_collection.get(
            where={"source": filename},
            include=["metadatas"]
        )

        if existing["ids"]:
            skipped_files.append(filename)
            continue

        pages = extract_pdf_pages(file_path)

        for page in pages:

            chunks = split_text(page["text"])

            for chunk in chunks:

                embedding = nexus_embedding(chunk)

                nexus_collection.add(
                    ids=[str(uuid.uuid4())],
                    documents=[chunk],
                    embeddings=[embedding],
                    metadatas=[{
                        "source": filename,
                        "page": page["page"]
                    }]
                )

                total_chunks += 1
                time.sleep(2)

    message = f"Succes

In [248]:
component_names = [
    "knowledge_button",
    "official_files",
    "upload_button",
    "personal_file",
    "assistant_button",
    "refresh_dashboard",
    "demo"
]

print("========== COMPONENT CHECK ==========")

for name in component_names:
    print(name, ":", name in globals())

========== COMPONENT CHECK ==========
knowledge_button : False
official_files : False
upload_button : True
personal_file : True
assistant_button : False
refresh_dashboard : False
demo : True


In [249]:
print("========== CURRENT DEMO ==========")

print("Demo type:", type(demo).__name__)
print("Gradio version:", gr.__version__)

print("\n========== EXISTING CALLBACKS ==========")

for fn_id, fn_info in demo.fns.items():
    fn = fn_info.fn

    print(
        fn_id,
        getattr(fn, "__name__", "Unknown"),
        "| Inputs:", len(fn_info.inputs),
        "| Outputs:", len(fn_info.outputs)
    )

print("\n========== RESTORED DOCUMENTS ==========")

try:
    print("Total:", len(documents))
    for doc in documents:
        print(doc)
except NameError:
    print("Documents variable is not currently defined.")

========== CURRENT DEMO ==========
Demo type: Blocks
Gradio version: 6.31.0

========== EXISTING CALLBACKS ==========
0 save_government_files | Inputs: 1 | Outputs: 1
1 upload_personal | Inputs: 2 | Outputs: 4
2 <lambda> | Inputs: 1 | Outputs: 1
3 renewal_callback | Inputs: 2 | Outputs: 1

========== RESTORED DOCUMENTS ==========
Total: 1
{'document': 'sample_certificate.pdf', 'type': 'Training Certificate', 'issue_date': '2026-11-01', 'expiry_date': '2027-11-01', 'status': 'Valid', 'confidence': ''}


In [ ]:
import gradio as gr
import pandas as pd
import json

def format_expiry_table(documents):
    rows = []
    for doc in documents:
        rows.append([
            doc.get("document", "Unknown"),
            doc.get("type", "Unknown"),
            doc.get("issue_date", "Not found"),
            doc.get("expiry_date", "Not applicable"),
            str(doc.get("days_remaining", "N/A")),
            doc.get("status", "Not available")
        ])
    return rows

def renewal_callback(document_type, question):
    if not question or not question.strip():
        return f"Please enter a question regarding your {document_type} renewal."
    try:
        prompt = f"Document Type: {document_type}\nQuestion: {question}"
        return rag_answer(prompt)
    except Exception as e:
        return f"Error fetching renewal guidance: {str(e)}"


def upload_personal(files, documents):

    if not files:
        return (
            "No files uploaded.",
            documents,
            format_expiry_table(check_expiry(documents)),
            generate_dashboard(documents)
        )

    if not isinstance(files, list):
        files = [files]

    successful = []
    failed = []

    for file in files:
        try:
            # Save original file permanently in Google Drive
            saved_path = save_personal_file(file)

            # Process the saved copy
            documents = process_personal_document(
                saved_path, documents
            )

            successful.append(os.path.basename(saved_path))

        except Exception as e:
            print(f"ERROR processing {file}: {str(e)}")
            failed.append(os.path.basename(file))

    expiry_results = check_expiry(documents)
    table = format_expiry_table(expiry_results)
    dashboard = generate_dashboard(documents)

    message = (
        f"Successfully processed: {len(successful)} document(s)."
    )

    if successful:
        message += "\n\nProcessed files:\n" + "\n".join(successful)

    if failed:
        message += "\n\nFailed files:\n" + "\n".join(failed)

    return message, documents, table, dashboard


def upload_official(files):
    if not files:
        return "Please upload official PDF guidelines."
    try:
        message = add_to_knowledge_base(files)
        return message
    except Exception as e:
        return f"Error: {str(e)}"

def ask_assistant(question, documents):
    if not question or not question.strip():
        return "Please enter a question."

    try:
        result = agent_assistant(question, documents or [])

        if isinstance(result, (dict, list)):
            return json.dumps(result, indent=2, ensure_ascii=False)

        return str(result)

    except Exception as e:
        return f"Agent Error: {str(e)}"

with gr.Blocks(title="AI Government Service Assistant") as demo:

    documents_state = gr.State(
        load_documents_from_database()
    )

    print("Restored documents:", len(documents_state.value))

    gr.HTML("""
    <h1>AI Personal Document & Government Service Assistant</h1>
    <p>Manage documents, monitor expiry dates, discover government schemes and get AI guidance.</p>
    """)

    # DASHBOARD
    with gr.Tab("Dashboard"):
        dashboard_output = gr.HTML(
            generate_dashboard(load_documents_from_database())
        )
        gr.HTML("<p>Your personal document management overview.</p>")

    # PERSONAL DOCUMENTS
    with gr.Tab("Personal Documents"):
        personal_file = gr.File(
            label="Upload Multiple Personal Document",
            file_types=[".pdf", ".png", ".jpg", ".jpeg"],
            file_count="multiple",
            type="filepath"
        )

        upload_button = gr.Button("Process Document")

        upload_message = gr.Textbox(
            label="Processing Status"
        )

        document_table = gr.Dataframe(
            headers=[
                "Document",
                "Type",
                "Issue Date",
                "Expiry Date",
                "Days Remaining",
                "Status"
            ],
            value=format_expiry_table(
                check_expiry(load_documents_from_database())
            ),
            label="My Documents",
            interactive=False
        )

    # RENEWAL GUIDANCE
    with gr.Tab("Renewal Guidance"):
        gr.HTML("<h2>AI Government Document Renewal Assistant</h2>")

        renewal_document = gr.Dropdown(
            choices=[
                "Passport",
                "PAN Card",
                "Aadhaar Card",
                "Scholarship Certificate",
                "Other"
            ],
            label="Select Document Type"
        )

        renewal_question = gr.Textbox(
            label="Ask your renewal question",
            placeholder="What are the steps to renew my passport?"
        )

        renewal_button = gr.Button("Get Renewal Guidance")

        renewal_answer = gr.Markdown()

    # GOVERNMENT KNOWLEDGE BASE
    with gr.Tab("Government Knowledge Base"):
        government_file = gr.File(
            label="Upload Multiple Government PDFs",
            file_types=[".pdf"],
            file_count="multiple",
            type="filepath"
        )

        government_upload_button = gr.Button(
            "Upload and Index Documents"
        )

        government_upload_status = gr.Textbox(
            label="Knowledge Base Status"
        )

        government_upload_button.click(
            fn=save_government_files,
            inputs=[government_file],
            outputs=[government_upload_status]
        )

    # AI ASSISTANT
    with gr.Tab("AI Assistant"):
        question = gr.Textbox(
            label="Ask your question",
            placeholder="Which scholarships might I be eligible for?"
        )

        ask_button = gr.Button("Ask AI Assistant")

        answer = gr.Markdown()

    with gr.Tab("✏️ Edit Documents"):
        gr.Markdown("## Edit Personal Document Details")
        gr.Markdown("Correct information extracted by Gemini AI.")

        edit_table = gr.Dataframe(
            headers=[
                "Document",
                "Type",
                "Issue Date",
                "Expiry Date",
                "Days Remaining",
                "Status"
            ],
            datatype=["str", "str", "str", "str", "str", "str"],
            interactive=True,
            label="Editable Document Details"
        )

        edit_status = gr.Textbox(
            label="Update Status",
            interactive=False
        )

        save_button = gr.Button(
            "💾 Save Changes",
            variant="primary"
        )

    with gr.Tab("🔍 Search Documents"):
        gr.Markdown("## Search & Filter Personal Documents")
        gr.Markdown("Find your documents by name, type or expiry status.")

        with gr.Row():
            search_input = gr.Textbox(
                label="Search Document",
                placeholder="Enter document name or type..."
            )

            status_filter = gr.Dropdown(
                choices=[
                    "All",
                    "Valid",
                    "Expiring Soon",
                    "Expired",
                    "Not applicable"
                ],
                value="All",
                label="Filter by Status"
            )

        search_button = gr.Button(
            "🔍 Search Documents",
            variant="primary"
        )

        search_results = gr.Dataframe(
            headers=[
                "Document",
                "Type",
                "Issue Date",
                "Expiry Date",
                "Days Remaining",
                "Status"
            ],
            interactive=False,
            label="Search Results"
        )

    with gr.Tab("🔔 Expiry Reminders"):
        gr.Markdown(
            """
            # 🔔 Document Expiry Reminder System

            Automatically monitor your personal documents
            and generate reminders before their expiry dates.

            ### Available Reminder Alerts

            - ⚠️ 7 Days Before Expiry
            - 🚨 3 Days Before Expiry
            - ❌ On Expiry Date
            """
        )

        # Expiry Monitoring
        gr.Markdown("## 1. Expiry Monitoring")

        check_button = gr.Button(
            "🔍 Check Document Expiry",
            variant="primary"
        )

        expiry_status = gr.Textbox(
            label="Monitoring Status",
            lines=3,
            interactive=False
        )

        expiry_results = gr.Dataframe(
            headers=[
                "Document",
                "Days Remaining",
                "Reminder Type",
                "Status"
            ],
            label="Newly Generated Reminders",
            interactive=False
        )

        gr.Markdown("---")

        # Reminder History
        gr.Markdown("## 2. Reminder History")

        history_button = gr.Button(
            "📋 View Reminder History"
        )

        reminder_history_table = gr.Dataframe(
            value=get_reminder_history(),
            headers=[
                "Reminder ID",
                "Document",
                "Expiry Date",
                "Reminder Type",
                "Status",
                "Created At"
            ],
            label="Reminder Records",
            interactive=False
        )

        gr.Markdown("---")

        # Notification Preview
        gr.Markdown("## 3. Notification Preview")

        preview_button = gr.Button(
            "📩 Generate Notification Preview",
            variant="primary"
        )

        notification_output = gr.Textbox(
            label="Notification Details",
            lines=12,
            interactive=False
        )

        gr.Markdown("---")

        # Dry Run Testing
        gr.Markdown("## 4. Test Reminder Notifications")
        gr.Markdown(
            """
            Generate notification previews without actually
            delivering emails.
            """
        )

        send_button = gr.Button(
            "🧪 Test Notifications",
            variant="secondary"
        )

        send_output = gr.Textbox(
            label="Test Results",
            lines=6,
            interactive=False
        )

    # ==================== REGISTER EVENTS ====================
    # All Gradio UI layout components are now instantiated, so variables exist.

    # Personal Documents tab actions
    upload_event = upload_button.click(
        upload_personal,
        inputs=[personal_file, documents_state],
        outputs=[
            upload_message,
            documents_state,
            document_table,
            dashboard_output
        ]
    )

    upload_event.then(
        lambda docs: format_expiry_table(check_expiry(docs)),
        inputs=[documents_state],
        outputs=[edit_table]
    )

    # Renewal Guidance tab actions
    renewal_button.click(
        renewal_callback,
        inputs=[
            renewal_document,
            renewal_question
        ],
        outputs=renewal_answer
    )


    # AI Assistant tab actions
    ask_button.click(
        fn=agent_assistant,
        inputs=[question, documents_state],
        outputs=answer
    )

    # Edit tab actions
    save_button.click(
        save_document_edits,
        inputs=[edit_table, documents_state],
        outputs=[
            edit_status,
            documents_state,
            document_table,
            dashboard_output
        ]
    )

    # Search tab actions
    search_button.click(
        search_documents,
        inputs=[
            search_input,
            status_filter,
            documents_state
        ],
        outputs=[search_results]
    )

    # Expiry Reminders tab actions
    check_button.click(
        fn=check_expiry_reminders,
        inputs=[documents_state],
        outputs=[
            expiry_status,
            expiry_results
        ]
    )

    history_button.click(
        fn=get_reminder_history,
        inputs=[],
        outputs=[reminder_history_table]
    )

    preview_button.click(
        fn=get_notification_previews,
        inputs=[],
        outputs=[notification_output]
    )

    send_button.click(
        fn=preview_send_reminders,
        inputs=[],
        outputs=[send_output]
    )

demo.launch(
    share=True,
    debug=True
)

Restored documents: 1
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://78a8117e7a859fac26.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR: 403 Client Error: Forbidden for url: https://nexusapi.navigatelabs.ai/v1/embeddings
